# 1\.3\.1 Create Merged Mobility Layer

By the end of \`1\.2\.4\`, the Taxi\-Zone\-compatible mobility modalities share a common temporal contract and a common analytical grain, and \`1\.2\.5\` / \`1\.2\.6\` add weather and static spatial context as companion layers\. This notebook uses that foundation to build the dense Taxi Zone x Date x Temporal Bucket mobility panel, validate the merged structure, and write the final downstream handoff tables\. Bridge & Tunnel, weather, and the Taxi Zone reference/context layers are carried forward as separate companion outputs rather than being forced into the core mobility panel\.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import gc
import time

import shutil
import json

import geopandas as gpd
from shapely import wkb

## 1\.3\.1\.1 Load and Inspect Harmonized Tables

Before we merge anything, we sanity\-check the harmonized outputs we are about to trust\. Even after \`1\.2\.4\`, the modalities still differ a lot in coverage and observation style: Traffic is sparse and sampled, while Bus, Subway, and TLC are much closer to full panel coverage\. This section confirms the working grain and coverage we need for the merge, then uses those diagnostics to choose between a sparse panel and a dense Taxi Zone x Date x Temporal Bucket spine\.

In [2]:
# ---------------------------------------------------------------------
# 1.3.1 Control Panel and Paths
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

HARMONIZED_INPUT_DIR = PIPELINE_DATA_DIR / "1.2.4.harmonized_mobility_datasets"
WEATHER_INPUT_DIR = PIPELINE_DATA_DIR / "1.2.5.harmonized_weather"
SPATIAL_CONTEXT_INPUT_DIR = PIPELINE_DATA_DIR / "1.2.6.final_tables"

MERGED_INTERMEDIATE_DIR = PIPELINE_DATA_DIR / "1.3.1.intermediate_tables"
MERGED_FINAL_DIR = PIPELINE_DATA_DIR / "1.3.1.final_tables"

MERGED_INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
MERGED_FINAL_DIR.mkdir(parents=True, exist_ok=True)

REBUILD_MERGED_MOBILITY_LAYER = False
REBUILD_ANALYSIS_READY_PANEL = False
RUN_MERGED_PANEL_VALIDATION = True

STUDY_START_DATE = "2023-01-01"
STUDY_END_DATE = "2026-03-31"
CONGESTION_PRICING_START_DATE = "2025-01-05"

traffic_path = HARMONIZED_INPUT_DIR / "traffic_mobility_table.parquet"
bus_path = HARMONIZED_INPUT_DIR / "bus_mobility_table.parquet"
subway_path = HARMONIZED_INPUT_DIR / "subway_mobility_table.parquet"
tlc_path = HARMONIZED_INPUT_DIR / "tlc_mobility_table.parquet"
bridge_tunnel_path = HARMONIZED_INPUT_DIR / "bridge_tunnel_mobility_table.parquet"
weather_context_path = WEATHER_INPUT_DIR / "weather_taxi_zone_context.parquet"

taxi_zones_path = SPATIAL_CONTEXT_INPUT_DIR / "taxi_zone_spatial_context.parquet"
taxi_zone_connectivity_path = SPATIAL_CONTEXT_INPUT_DIR / "taxi_zone_connectivity.parquet"

tlc_harmonized_input_path = tlc_path

merged_mobility_layer_path = MERGED_INTERMEDIATE_DIR / "merged_mobility_layer.parquet"
fixed_merged_mobility_layer_path = merged_mobility_layer_path

audit_only_output_path = MERGED_FINAL_DIR / "audit_only_mobility_panel.parquet"
analysis_ready_output_path = MERGED_FINAL_DIR / "analysis_ready_mobility_panel.parquet"
bridge_tunnel_output_path = MERGED_FINAL_DIR / "bridge_tunnel_mobility_panel.parquet"
weather_context_output_path = MERGED_FINAL_DIR / "weather_taxi_zone_context.parquet"
taxi_zones_output_path = MERGED_FINAL_DIR / "nyc_taxi_zones_harmonized.parquet"
taxi_zone_connectivity_output_path = MERGED_FINAL_DIR / "taxi_zone_connectivity.parquet"
merged_panel_contract_path = MERGED_FINAL_DIR / "merged_mobility_panel_contract.json"

paths_to_check = [
    traffic_path,
    bus_path,
    subway_path,
    tlc_path,
    bridge_tunnel_path,
    taxi_zones_path,
    taxi_zone_connectivity_path,
    weather_context_path,
]

for path in paths_to_check:
    print(path, "EXISTS" if path.exists() else "MISSING")

con = duckdb.connect()

KeyboardInterrupt: 

In [3]:
duckdb.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{tlc_path.as_posix()}')
""").df()

,column_name,column_type,null,key,default,extra
0,taxi_zone_id,INTEGER,YES,None,None,None
1,date,DATE,YES,None,None,None
2,year,BIGINT,YES,None,None,None
3,month,BIGINT,YES,None,None,None
4,day_of_week,INTEGER,YES,None,None,None
5,daypart,VARCHAR,YES,None,None,None
6,daypart_order,INTEGER,YES,None,None,None
7,temporal_bucket,VARCHAR,YES,None,None,None
8,temporal_bucket_order,INTEGER,YES,None,None,None
9,observation_sequence_id,BIGINT,YES,None,None,None


At a high level, there are two reasonable ways to construct the integrated mobility layer\. A sparse panel would retain only Taxi Zone × Date × Temporal Bucket combinations that appear in at least one mobility dataset\. This produces a smaller table and naturally reflects source\-data availability, particularly for Traffic\. A dense panel instead creates the full analytical universe first and then joins mobility features onto it, leaving unobserved modality values as nulls\. The dense approach produces a larger table but makes the project's core analytical unit explicit and greatly simplifies downstream feature engineering, missingness analysis, anomaly detection, clustering, and forecasting workflows\. Let's do some diagnostics below to quantify both options so that the final design choice is based on the observed data rather than assumptions\.

In [4]:
panel_strategy_comparison_df = con.execute(f"""
    WITH all_observed_keys AS (
        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{traffic_path}')

        UNION

        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{bus_path}')

        UNION

        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{subway_path}')

        UNION

        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{tlc_path}')
    ),

    date_spine AS (
        SELECT date
        FROM generate_series(
            DATE '2023-01-01',
            DATE '2026-03-31',
            INTERVAL 1 DAY
        ) AS t(date)
    ),

    temporal_buckets AS (
        SELECT DISTINCT temporal_bucket
        FROM all_observed_keys
    ),

    panel_zones AS (
        SELECT DISTINCT taxi_zone_id
        FROM all_observed_keys
    ),

    dense_panel AS (
        SELECT
            z.taxi_zone_id,
            d.date,
            b.temporal_bucket
        FROM panel_zones z
        CROSS JOIN date_spine d
        CROSS JOIN temporal_buckets b
    )

    SELECT
        (SELECT COUNT(*) FROM all_observed_keys) AS sparse_panel_rows,
        (SELECT COUNT(*) FROM dense_panel) AS dense_panel_rows,
        (SELECT COUNT(*) FROM dense_panel) - (SELECT COUNT(*) FROM all_observed_keys) AS additional_dense_rows,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM all_observed_keys) / (SELECT COUNT(*) FROM dense_panel),
            2
        ) AS sparse_panel_pct_of_dense,
        ROUND(
            100.0 * ((SELECT COUNT(*) FROM dense_panel) - (SELECT COUNT(*) FROM all_observed_keys)) / (SELECT COUNT(*) FROM dense_panel),
            2
        ) AS dense_unobserved_key_pct,
        (SELECT COUNT(DISTINCT taxi_zone_id) FROM all_observed_keys) AS taxi_zone_count,
        (SELECT COUNT(DISTINCT date) FROM all_observed_keys) AS observed_date_count,
        (SELECT COUNT(DISTINCT temporal_bucket) FROM all_observed_keys) AS temporal_bucket_count
""").df()

display(panel_strategy_comparison_df)

,sparse_panel_rows,dense_panel_rows,additional_dense_rows,sparse_panel_pct_of_dense,dense_unobserved_key_pct,taxi_zone_count,observed_date_count,temporal_bucket_count
0,1546012,3119180,1573168,49.56,50.44,263,1186,10


Findings\. The sparse panel contains approximately 1\.55 million observed mobility states, while the fully dense Taxi Zone × Date × Temporal Bucket universe contains approximately 3\.12 million possible states\. In other words, observed mobility records occupy about 50% of the analytical space, with the remaining 50% representing combinations where no modality reported activity\. Although the dense panel is roughly twice the size of the sparse panel, it remains small enough to manage comfortably in DuckDB while providing a complete and explicit analytical framework\. Because the project's analytical unit is defined as Taxi Zone × Date × Temporal Bucket, the dense panel was selected as the foundation for the merged mobility layer, allowing modality\-specific missingness to remain visible rather than being hidden through row omission\.

## 1\.3\.1\.2 Define Merge Strategy

Now that we've selected the dense\-panel strategy, what does our actual merge plan look like? The merged mobility layer will be built at the Taxi Zone × Date × Temporal Bucket grain using the harmonized Taxi Zone reference layer, the full study\-period date spine, and the project’s 10 temporal buckets\. Traffic, Bus, Subway, and TLC will be left joined onto that base using taxi\_zone\_id, date, and temporal\_bucket\. All modality fields produced in 1\.2\.4 will be retained\. Bridge & Tunnel will remain separate because its natural grain is Facility × Direction × Date × Temporal Bucket, not Taxi Zone × Date × Temporal Bucket\. Null values in the merged panel are expected and meaningful: they indicate that a given modality did not observe that Taxi Zone/date/bucket combination, not that the Taxi Zone/date/bucket itself is invalid\.

### Fields to Retain

All harmonized modality fields will be retained at this stage\. 

### Merge Architecture

The merged mobility layer will use a dense Taxi Zone × Date × Temporal Bucket panel as the base\. The panel will be built from all harmonized Taxi Zones, all dates in the study period, and all 10 project temporal buckets\. Traffic, Bus, Subway, and TLC will be left joined onto this base using taxi\_zone\_id, date, and temporal\_bucket\. Taxi Zone metadata will be attached from the harmonized Taxi Zone reference layer using taxi\_zone\_id = location\_id\.

### Expected Final Merged Schema

The final merged mobility layer will contain one row per Taxi Zone × Date × Temporal Bucket mobility state\. The primary keys and descriptive fields will be: taxi\_zone\_id, zone, borough, canonical\_location\_id, date, year, month, day\_of\_week, daypart, daypart\_order, temporal\_bucket, temporal\_bucket\_order, observation\_sequence\_id, and pre\_post\_cp\.

Traffic features: traffic\_volume, traffic\_observation\_count, traffic\_distinct\_segment\_count\.

Bus features: avg\_bus\_speed, avg\_bus\_travel\_time, bus\_speed\_stddev, bus\_travel\_time\_stddev, bus\_trip\_count, bus\_segment\_observation\_count, bus\_assignment\_method\_count\.

Subway features: subway\_ridership, subway\_transfers, subway\_observation\_count, subway\_station\_complex\_count\.

TLC features: taxi\_trip\_count, taxi\_avg\_trip\_distance, taxi\_avg\_trip\_duration, taxi\_avg\_trip\_speed, taxi\_distinct\_dropoff\_zone\_count, taxi\_source\_od\_row\_count, fhvhv\_trip\_count, fhvhv\_avg\_trip\_distance, fhvhv\_avg\_trip\_duration, fhvhv\_avg\_trip\_speed, fhvhv\_distinct\_dropoff\_zone\_count, fhvhv\_source\_od\_row\_count, tlc\_modality\_count, and tlc\_contract\_row\_count\. These fields now come directly from the 1\.2\.4 harmonized TLC table, which consumes the final 1\.2\.3 Taxi/FHVHV contract\.

Conceptually, the final output will resemble:

taxi\_zone\_id	 borough	          date	         temporal\_bucket	   traffic\_volume	    avg\_bus\_speed	  subway\_ridership	taxi\_trip\_count	fhvhv\_trip\_count   \.\.\.
161	                 Manhattan	2025\-02\-14	weekday\_pm\_peak	14,230	                   8\.4	                20,122	                8,321	                19,442                  \.\.\.
236	                 Manhattan	2025\-02\-14	weekday\_pm\_peak	NaN  	                   9\.1	                        17,811	                7,210	                15,908                  \.\.\.
74	                 Brooklyn	2025\-02\-14	weekday\_pm\_peak	6,210	                   11\.7	                5,511	                2,122	                4,388                  \.\.\.

The actual table will contain the full set of retained Traffic, Bus, Subway, and TLC mobility measures\. TLC now arrives with the final upstream design: Yellow and Green have already been merged into Taxi using valid aggregate numerators and denominators, FHVHV remains a separate modality, and Taxi destination diversity has already been computed before the pickup\-temporal collapse\. Each row still represents one Taxi Zone × Date × Temporal Bucket mobility state, and null values remain meaningful: they represent modality\-specific non\-observation within an otherwise valid panel row\.

The merged layer will include the common panel fields, Taxi Zone metadata, and all retained mobility measures from the four Taxi\-Zone\-compatible modalities\. The expected structure is: taxi\_zone\_id, zone, borough, canonical\_location\_id, date, year, month, day\_of\_week, temporal\_bucket, pre\_post\_cp, Traffic features, Bus features, Subway features, and TLC features\. Null values are expected and meaningful: they represent modality\-specific non\-observation within an otherwise valid Taxi Zone × Date × Temporal Bucket state\.

## 1\.3\.1\.3 Merge Dense Mobility Panel

Having selected a dense\-panel architecture, we can now construct the merged mobility layer\. The process consists of three steps: building the Taxi Zone × Date × Temporal Bucket base panel, attaching Taxi Zone metadata from the harmonized reference layer, and then left joining Traffic, Bus, Subway, and TLC mobility measures\. Because all four modalities were validated in 1\.2\.4 and confirmed to share a common analytical grain in 1\.3\.1, the merge can be performed directly without additional aggregation\.

### Build Dense Mobility Spine

Let's create the base analytical universe for the merged layer\. This spine contains every Taxi Zone observed in the harmonized mobility tables, every date in the study period, and the valid temporal buckets for that date type\. Weekdays only receive weekday\_\* buckets, and weekends only receive weekend\_\* buckets\. No modality metrics are joined yet; this is just the clean Taxi Zone × Date × Temporal Bucket scaffold\.

In [5]:
# ---------------------------------------------------------------------
# Build dense Taxi Zone x Date x Temporal Bucket mobility spine
# ---------------------------------------------------------------------

# The dense spine is the canonical analytical universe for the merged panel.
# Modality values are joined later, so missing observations remain visible as nulls.
dense_mobility_spine_df = con.execute(f"""
    WITH observed_taxi_zones AS (
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{traffic_path.as_posix()}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{bus_path.as_posix()}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{subway_path.as_posix()}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{tlc_path.as_posix()}')
    ),

    date_spine AS (
        SELECT
            date,
            STRFTIME(date, '%A') AS day_of_week
        FROM generate_series(
            DATE '{STUDY_START_DATE}',
            DATE '{STUDY_END_DATE}',
            INTERVAL 1 DAY
        ) AS t(date)
    ),

    temporal_bucket_order AS (
        SELECT *
        FROM (VALUES
            ('weekday_overnight', 'overnight', 1, 1),
            ('weekday_am_peak', 'am_peak', 2, 2),
            ('weekday_midday', 'midday', 3, 3),
            ('weekday_pm_peak', 'pm_peak', 4, 4),
            ('weekday_evening', 'evening', 5, 5),
            ('weekend_overnight', 'overnight', 1, 6),
            ('weekend_am_peak', 'am_peak', 2, 7),
            ('weekend_midday', 'midday', 3, 8),
            ('weekend_pm_peak', 'pm_peak', 4, 9),
            ('weekend_evening', 'evening', 5, 10)
        ) AS t(temporal_bucket, daypart, daypart_order, temporal_bucket_order)
    ),

    observed_temporal_buckets AS (
        SELECT DISTINCT temporal_bucket FROM read_parquet('{traffic_path.as_posix()}')
        UNION
        SELECT DISTINCT temporal_bucket FROM read_parquet('{bus_path.as_posix()}')
        UNION
        SELECT DISTINCT temporal_bucket FROM read_parquet('{subway_path.as_posix()}')
        UNION
        SELECT DISTINCT temporal_bucket FROM read_parquet('{tlc_path.as_posix()}')
    ),

    valid_date_buckets AS (
        SELECT
            d.date,
            d.day_of_week,
            b.temporal_bucket,
            b.daypart,
            b.daypart_order,
            b.temporal_bucket_order
        FROM date_spine d
        CROSS JOIN temporal_bucket_order b
        INNER JOIN observed_temporal_buckets observed
            ON b.temporal_bucket = observed.temporal_bucket
        WHERE
            (
                d.day_of_week IN ('Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday')
                AND b.temporal_bucket LIKE 'weekday_%'
            )
            OR
            (
                d.day_of_week IN ('Saturday', 'Sunday')
                AND b.temporal_bucket LIKE 'weekend_%'
            )
    ),

    date_bucket_sequence AS (
        SELECT
            *,
            DENSE_RANK() OVER (
                ORDER BY date, temporal_bucket_order
            ) AS observation_sequence_id
        FROM valid_date_buckets
    )

    SELECT
        z.taxi_zone_id,
        v.date,
        EXTRACT(year FROM v.date)::INTEGER AS year,
        EXTRACT(month FROM v.date)::INTEGER AS month,
        v.day_of_week,
        v.daypart,
        v.daypart_order,
        v.temporal_bucket,
        v.temporal_bucket_order,
        v.observation_sequence_id,
        CASE
            WHEN v.date >= DATE '{CONGESTION_PRICING_START_DATE}' THEN 'post_cp'
            ELSE 'pre_cp'
        END AS pre_post_cp
    FROM observed_taxi_zones z
    CROSS JOIN date_bucket_sequence v
    ORDER BY
        z.taxi_zone_id,
        v.date,
        v.temporal_bucket_order
""").df()

display(dense_mobility_spine_df.head())
print(f"Dense mobility spine rows: {len(dense_mobility_spine_df):,}")

,taxi_zone_id,date,year,month,day_of_week,daypart,daypart_order,temporal_bucket,temporal_bucket_order,observation_sequence_id,pre_post_cp
0,1,2023-01-01,2023,1,Sunday,overnight,1,weekend_overnight,6,1,pre_cp
1,1,2023-01-01,2023,1,Sunday,am_peak,2,weekend_am_peak,7,2,pre_cp
2,1,2023-01-01,2023,1,Sunday,midday,3,weekend_midday,8,3,pre_cp
3,1,2023-01-01,2023,1,Sunday,pm_peak,4,weekend_pm_peak,9,4,pre_cp
4,1,2023-01-01,2023,1,Sunday,evening,5,weekend_evening,10,5,pre_cp


Dense mobility spine rows: 1,559,590


### Attach Taxi Zone Metadata \(borough\)

Next, let's enrich the dense spine with Taxi Zone metadata from the harmonized reference layer\. The mobility tables use taxi\_zone\_id, while the Taxi Zone reference layer uses location\_id, so the join maps spine\.taxi\_zone\_id to zones\.location\_id\. This adds the readable zone name, borough, and canonical location ID before we attach modality metrics\.

In [6]:
# ---------------------------------------------------------------------
# Attach Taxi Zone spatial context to the dense mobility spine
# ---------------------------------------------------------------------

dense_mobility_panel_base_df = con.execute(f"""
    WITH taxi_zone_metadata AS (
        SELECT
            location_id AS taxi_zone_id,
            zone,
            borough,
            canonical_location_id,
            centroid_x,
            centroid_y,
            is_cbd_zone,
            is_cbd_gateway_zone,
            is_cbd_adjacent_zone,
            cbd_spatial_category,
            distance_to_cbd_feet,
            distance_to_cbd_miles,
            distance_to_gateway_feet,
            distance_to_gateway_miles,
            connected_neighbor_count,
            network_betweenness_centrality,
            network_closeness_centrality,
            network_clustering_coefficient
        FROM read_parquet('{taxi_zones_path.as_posix()}')
    )

    SELECT
        spine.taxi_zone_id,
        zones.zone,
        zones.borough,
        zones.canonical_location_id,
        zones.centroid_x,
        zones.centroid_y,
        zones.is_cbd_zone,
        zones.is_cbd_gateway_zone,
        zones.is_cbd_adjacent_zone,
        zones.cbd_spatial_category,
        zones.distance_to_cbd_feet,
        zones.distance_to_cbd_miles,
        zones.distance_to_gateway_feet,
        zones.distance_to_gateway_miles,
        zones.connected_neighbor_count,
        zones.network_betweenness_centrality,
        zones.network_closeness_centrality,
        zones.network_clustering_coefficient,
        spine.date,
        spine.year,
        spine.month,
        spine.day_of_week,
        spine.daypart,
        spine.daypart_order,
        spine.temporal_bucket,
        spine.temporal_bucket_order,
        spine.observation_sequence_id,
        spine.pre_post_cp
    FROM dense_mobility_spine_df spine
    LEFT JOIN taxi_zone_metadata zones
        ON spine.taxi_zone_id = zones.taxi_zone_id
    ORDER BY spine.taxi_zone_id, spine.date, spine.temporal_bucket_order
""").df()

display(dense_mobility_panel_base_df.head())
print(f"Dense mobility panel base rows: {len(dense_mobility_panel_base_df):,}")

,taxi_zone_id,zone,borough,canonical_location_id,centroid_x,centroid_y,is_cbd_zone,is_cbd_gateway_zone,is_cbd_adjacent_zone,cbd_spatial_category,...,date,year,month,day_of_week,daypart,daypart_order,temporal_bucket,temporal_bucket_order,observation_sequence_id,pre_post_cp
0,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,2023-01-01,2023,1,Sunday,overnight,1,weekend_overnight,6,1,pre_cp
1,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,2023-01-01,2023,1,Sunday,am_peak,2,weekend_am_peak,7,2,pre_cp
2,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,2023-01-01,2023,1,Sunday,midday,3,weekend_midday,8,3,pre_cp
3,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,2023-01-01,2023,1,Sunday,pm_peak,4,weekend_pm_peak,9,4,pre_cp
4,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,2023-01-01,2023,1,Sunday,evening,5,weekend_evening,10,5,pre_cp


Dense mobility panel base rows: 1,559,590


Findings\. The metadata join worked cleanly\. We now have zone names, boroughs, and canonical location IDs attached to every Taxi Zone × Date × Temporal Bucket mobility state, giving us a much more interpretable panel to work with as we start attaching the mobility metrics\.

Let's do a quick QA confirms that the base panel still has the expected grain after metadata has been attached and checks whether any Taxi Zone rows failed to resolve to borough metadata\.

In [7]:
dense_panel_base_qa_df = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS distinct_grain_count,
        COUNT(*) - COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS duplicate_grain_rows,
        COUNT(DISTINCT taxi_zone_id) AS taxi_zone_count,
        COUNT(DISTINCT date) AS date_count,
        COUNT(DISTINCT temporal_bucket) AS temporal_bucket_count,
        COUNT(DISTINCT observation_sequence_id) AS observation_sequence_count,
        SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) AS rows_missing_borough,
        SUM(CASE WHEN daypart IS NULL THEN 1 ELSE 0 END) AS rows_missing_daypart,
        SUM(CASE WHEN daypart_order IS NULL THEN 1 ELSE 0 END) AS rows_missing_daypart_order,
        SUM(CASE WHEN temporal_bucket_order IS NULL THEN 1 ELSE 0 END) AS rows_missing_temporal_bucket_order,
        SUM(CASE WHEN observation_sequence_id IS NULL THEN 1 ELSE 0 END) AS rows_missing_observation_sequence
    FROM dense_mobility_panel_base_df
""").df()

display(dense_panel_base_qa_df)

failed_dense_base_checks = dense_panel_base_qa_df.loc[
    (
        (dense_panel_base_qa_df["duplicate_grain_rows"] != 0)
        | (dense_panel_base_qa_df["rows_missing_borough"] != 0)
        | (dense_panel_base_qa_df["rows_missing_daypart"] != 0)
        | (dense_panel_base_qa_df["rows_missing_daypart_order"] != 0)
        | (dense_panel_base_qa_df["rows_missing_temporal_bucket_order"] != 0)
        | (dense_panel_base_qa_df["rows_missing_observation_sequence"] != 0)
    )
]

if not failed_dense_base_checks.empty:
    raise AssertionError(
        "Dense mobility panel base QA failed. Review grain, Taxi Zone metadata, or temporal ordering fields."
    )

,row_count,distinct_grain_count,duplicate_grain_rows,taxi_zone_count,date_count,temporal_bucket_count,observation_sequence_count,rows_missing_borough,rows_missing_daypart,rows_missing_daypart_order,rows_missing_temporal_bucket_order,rows_missing_observation_sequence
0,1559590,1559590,0,263,1186,10,5930,0.0,0.0,0.0,0.0,0.0


Findings: The panel returned to the expected analytical grain with exactly 1,559,590 unique Taxi Zone × Date × Temporal Bucket mobility states and no duplicate records\. We also confirmed that all 263 observed Taxi Zones successfully resolved to borough metadata, giving us a clean and fully attributed mobility spine before we start attaching Traffic, Bus, Subway, and TLC metrics\.

Before moving on, let's do one more sanity check on the Taxi Zone universe used to build the dense panel\. During harmonization, we introduced a handful of alias and special\-case Taxi Zone records to handle multipart geographies such as Corona and Governor's Island / Ellis Island / Liberty Island\. The question is whether the dense panel should be built from every Taxi Zone in the reference layer or only the Taxi Zones that actually appear in the harmonized mobility datasets\. The diagnostic below compares those two universes and highlights any mismatches\.

In [8]:
zone_universe_comparison_df = con.execute(f"""
    WITH observed_zones AS (
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{traffic_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{bus_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{subway_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{tlc_path}')
    ),

    reference_zones AS (
        SELECT DISTINCT location_id AS taxi_zone_id
        FROM read_parquet('{taxi_zones_path}')
    )

    SELECT
        COALESCE(o.taxi_zone_id, r.taxi_zone_id) AS taxi_zone_id,
        CASE WHEN o.taxi_zone_id IS NOT NULL THEN 1 ELSE 0 END AS in_observed_mobility,
        CASE WHEN r.taxi_zone_id IS NOT NULL THEN 1 ELSE 0 END AS in_reference_layer
    FROM observed_zones o
    FULL OUTER JOIN reference_zones r
        ON o.taxi_zone_id = r.taxi_zone_id
    WHERE
        o.taxi_zone_id IS NULL
        OR r.taxi_zone_id IS NULL
        OR COALESCE(o.taxi_zone_id, r.taxi_zone_id) IN (56, 57, 103, 104, 105)
    ORDER BY taxi_zone_id
""").df()

display(zone_universe_comparison_df)

,taxi_zone_id,in_observed_mobility,in_reference_layer
0,56,1,1
1,57,1,1
2,103,0,1
3,105,1,1


Findings: The comparison confirms that the dense panel is being built from the observed mobility universe rather than every Taxi Zone polygon in the reference layer\. This is the behavior we want\. Taxi Zones 56, 57, and 105 all appear in the harmonized mobility datasets and are therefore included in the panel\. Taxi Zone 103 exists in the spatial reference layer but never appears in any harmonized mobility output, so including it would create an entirely empty mobility state with no supporting observations\. As a result, the merged mobility layer uses the set of Taxi Zones actually represented in the mobility data while still preserving the reference\-layer enhancements made during harmonization\.

## 1\.3\.1\.4 Join Harmonized Mobility Metrics

We can now attach the harmonized mobility measures from each transportation modality\. Because all four datasets were standardized to the same analytical grain during 1\.2\.4 and validated in 1\.3\.1, the joins can be performed directly using taxi\_zone\_id, date, and temporal\_bucket\. We use left joins so that the dense panel remains intact even when a particular modality was not observed for a given mobility state\.

In [9]:
merged_mobility_layer_path = MERGED_INTERMEDIATE_DIR / "merged_mobility_layer.parquet"

should_build_merged_layer = (
    REBUILD_MERGED_MOBILITY_LAYER
    or not merged_mobility_layer_path.exists()
)

print(f"Rebuild merged mobility layer: {REBUILD_MERGED_MOBILITY_LAYER}")
print(f"Merged mobility layer exists: {merged_mobility_layer_path.exists()}")
print(f"Will build merged mobility layer: {should_build_merged_layer}")

Rebuild merged mobility layer: True
Merged mobility layer exists: True
Will build merged mobility layer: True


In [10]:
# ---------------------------------------------------------------------
# Build merged mobility layer with static spatial context
# ---------------------------------------------------------------------

if REBUILD_MERGED_MOBILITY_LAYER or not merged_mobility_layer_path.exists():
    con.register("dense_mobility_panel_base_df", dense_mobility_panel_base_df)

    con.execute(f"""
        COPY (
            SELECT
                base.*,
                traffic.traffic_volume,
                traffic.traffic_observation_count,
                traffic.traffic_distinct_segment_count,
                bus.avg_bus_speed,
                bus.avg_bus_travel_time,
                bus.bus_speed_stddev,
                bus.bus_travel_time_stddev,
                bus.bus_trip_count,
                bus.bus_segment_observation_count,
                bus.bus_assignment_method_count,
                subway.subway_ridership,
                subway.subway_transfers,
                subway.subway_observation_count,
                subway.subway_station_complex_count,
                tlc.taxi_trip_count,
                tlc.taxi_avg_trip_distance,
                tlc.taxi_avg_trip_duration,
                tlc.taxi_avg_trip_speed,
                tlc.taxi_distinct_dropoff_zone_count,
                tlc.taxi_source_od_row_count,
                tlc.fhvhv_trip_count,
                tlc.fhvhv_avg_trip_distance,
                tlc.fhvhv_avg_trip_duration,
                tlc.fhvhv_avg_trip_speed,
                tlc.fhvhv_distinct_dropoff_zone_count,
                tlc.fhvhv_source_od_row_count,
                tlc.tlc_modality_count,
                tlc.tlc_contract_row_count,
                tlc.spatial_assignment_method AS tlc_spatial_assignment_method,
                tlc.spatial_assignment_source AS tlc_spatial_assignment_source,
                tlc.spatial_assignment_confidence AS tlc_spatial_assignment_confidence,
                tlc.canonical_geography AS tlc_canonical_geography
            FROM dense_mobility_panel_base_df AS base
            LEFT JOIN read_parquet('{traffic_path.as_posix()}') AS traffic
                USING (taxi_zone_id, date, temporal_bucket)
            LEFT JOIN read_parquet('{bus_path.as_posix()}') AS bus
                USING (taxi_zone_id, date, temporal_bucket)
            LEFT JOIN read_parquet('{subway_path.as_posix()}') AS subway
                USING (taxi_zone_id, date, temporal_bucket)
            LEFT JOIN read_parquet('{tlc_path.as_posix()}') AS tlc
                USING (taxi_zone_id, date, temporal_bucket)
        )
        TO '{merged_mobility_layer_path.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

print(f"Merged mobility layer: {merged_mobility_layer_path}")

Merged mobility layer: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer.parquet


Let’s do a quick QA to make sure the joins didn’t multiply rows or break the panel grain\.

In [11]:
merged_mobility_preview_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{merged_mobility_layer_path}')
    LIMIT 10
""").df()

display(merged_mobility_preview_df)

,taxi_zone_id,zone,borough,canonical_location_id,centroid_x,centroid_y,is_cbd_zone,is_cbd_gateway_zone,is_cbd_adjacent_zone,cbd_spatial_category,...,fhvhv_avg_trip_duration,fhvhv_avg_trip_speed,fhvhv_distinct_dropoff_zone_count,fhvhv_source_od_row_count,tlc_modality_count,tlc_contract_row_count,tlc_spatial_assignment_method,tlc_spatial_assignment_source,tlc_spatial_assignment_confidence,tlc_canonical_geography
0,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
1,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
2,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
3,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
4,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
5,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
6,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
7,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
8,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
9,1,Newark Airport,EWR,1,935996.821016,191376.749531,0.0,1.0,0.0,gateway_to_cbd,...,NaN,NaN,<NA>,0.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone


Findings: The preview looks right\. We now have one row per Taxi Zone × Date × Temporal Bucket, with the shared panel fields up front and the modality metrics attached after that\. Newark Airport is a useful example because it shows why the dense panel matters: Traffic, Bus, and Subway are null for these rows, but TLC activity is still present\. That tells us the row is a valid mobility state even though only one modality observed activity there\. This is exactly the behavior we wanted from the merged layer: preserve the full panel, keep all 1\.2\.4 metrics, and let modality\-specific missingness stay visible instead of dropping the row\.

Let's inspect the troublesome zones 56 and 57\.

In [12]:
merged_mobility_preview_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{merged_mobility_layer_path}')
    WHERE taxi_zone_id IN (56, 57)
    ORDER BY date
    LIMIT 50
""").df()

display(merged_mobility_preview_df)

,taxi_zone_id,zone,borough,canonical_location_id,centroid_x,centroid_y,is_cbd_zone,is_cbd_gateway_zone,is_cbd_adjacent_zone,cbd_spatial_category,...,fhvhv_avg_trip_duration,fhvhv_avg_trip_speed,fhvhv_distinct_dropoff_zone_count,fhvhv_source_od_row_count,tlc_modality_count,tlc_contract_row_count,tlc_spatial_assignment_method,tlc_spatial_assignment_source,tlc_spatial_assignment_confidence,tlc_canonical_geography
0,56,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,833.639775,19.629994,133,351.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
1,57,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,732.535714,18.816586,26,41.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
2,56,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,834.835590,18.743805,108,236.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
3,56,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,1019.107595,18.820345,89,170.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
4,56,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,944.033951,20.230958,92,167.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
5,56,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,794.740331,23.349209,101,177.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
6,57,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,863.777778,17.607769,12,16.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
7,57,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,757.187500,16.669055,11,15.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
8,57,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,896.085714,24.319485,25,33.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
9,57,Corona,Queens,56,1.023497e+06,209766.516263,0.0,0.0,0.0,non_cbd,...,805.000000,28.450003,17,19.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone


Findings\. The Corona alias zones were preserved correctly through the merge process\. Zone 56 contains multimodal activity, while Zone 57 remains a TLC\-only zone, confirming that the special\-case spatial harmonization logic continues to behave as expected in the final merged layer\.

## 1\.3\.1\.5 Validate Merged Layer

Now that the merged mobility layer has been written, let's treat it like a final output instead of an intermediate table\. We want to confirm that the file exists, the grain is still clean, the dense panel has the expected shape, the spatial metadata resolved correctly, and the modality\-level missingness looks reasonable\. This section is intentionally lightweight: we are not doing EDA yet, and we are not judging whether the patterns are interesting\. We are just making sure the merged layer is structurally sound and ready for 1\.4\.

First, let's confirm that the merged output exists and get a quick file\-size check\.

In [13]:
print("Merged mobility layer path:", merged_mobility_layer_path)
print("Exists:", merged_mobility_layer_path.exists())

if merged_mobility_layer_path.exists():
    print(f"Size MB: {merged_mobility_layer_path.stat().st_size / (1024 * 1024):,.2f}")

Merged mobility layer path: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer.parquet
Exists: True
Size MB: 75.42


Let's start with the most important QA: one row per Taxi Zone × Date × Temporal Bucket\. This should match the dense panel size we selected earlier\.

In [14]:
merged_layer_grain_qa_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS distinct_grain_count,
        COUNT(*) - COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS duplicate_grain_rows,
        COUNT(DISTINCT taxi_zone_id) AS taxi_zone_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT date) AS date_count,
        COUNT(DISTINCT temporal_bucket) AS temporal_bucket_count,
        COUNT(DISTINCT pre_post_cp) AS pre_post_cp_count,
        SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) AS rows_missing_borough,
        SUM(CASE WHEN zone IS NULL THEN 1 ELSE 0 END) AS rows_missing_zone
    FROM read_parquet('{merged_mobility_layer_path}')
""").df()

display(merged_layer_grain_qa_df)

,row_count,distinct_grain_count,duplicate_grain_rows,taxi_zone_count,min_date,max_date,date_count,temporal_bucket_count,pre_post_cp_count,rows_missing_borough,rows_missing_zone
0,1559590,1559590,0,263,2023-01-01,2026-03-31,1186,10,2,0.0,0.0


Findings\. Everything looks good\. The merged layer contains 1\.55 million mobility states with zero duplicate grain rows, covers all 263 Taxi Zones, spans January 2023 through March 2026, and includes all 10 temporal buckets\. We also successfully resolved every row to Taxi Zone metadata, with no missing boroughs or zone names\.

Next, let's check how much of the dense panel is covered by each modality\. Nulls are expected here, especially for Traffic, but we want the coverage pattern to line up with what we learned from the harmonized inputs\.

In [15]:
# ---------------------------------------------------------------------
# Modality coverage summary for merged mobility layer
# ---------------------------------------------------------------------

modality_coverage_summary_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        ROUND(100.0 * SUM(CASE WHEN traffic_volume IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_traffic,
        ROUND(100.0 * SUM(CASE WHEN bus_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_bus,
        ROUND(100.0 * SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_subway,
        ROUND(100.0 * SUM(CASE WHEN taxi_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_taxi,
        ROUND(100.0 * SUM(CASE WHEN fhvhv_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_fhvhv
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

display(modality_coverage_summary_df)

,row_count,pct_with_traffic,pct_with_bus,pct_with_subway,pct_with_taxi,pct_with_fhvhv
0,1559590,0.54,94.67,58.49,98.2,98.2


Findings\. The modality coverage patterns look exactly like we'd expect from the harmonized inputs\. TLC and Bus provide the broadest coverage, each populating roughly half of the dense panel, while Subway covers about 58% of mobility states\. Traffic remains the sparsest modality at just 0\.54% of panel rows, reflecting the limited footprint of the roadway sensor network rather than any merge issue\. Most importantly, the merge preserved the expected coverage characteristics of each mobility system\.

Let's also check coverage by borough\. This gives us a quick read on whether any borough metadata or modality coverage looks obviously broken after the merge\.

In [16]:
# ---------------------------------------------------------------------
# TLC coverage summary using final Taxi/FHVHV fields
# ---------------------------------------------------------------------

tlc_coverage_summary_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        SUM(CASE WHEN taxi_trip_count IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_taxi,
        ROUND(100.0 * SUM(CASE WHEN taxi_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_taxi,
        SUM(CASE WHEN fhvhv_trip_count IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_fhvhv,
        ROUND(100.0 * SUM(CASE WHEN fhvhv_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_fhvhv,
        SUM(CASE WHEN taxi_trip_count IS NOT NULL OR fhvhv_trip_count IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_any_tlc,
        ROUND(
            100.0 * SUM(CASE WHEN taxi_trip_count IS NOT NULL OR fhvhv_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS pct_with_any_tlc
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

display(tlc_coverage_summary_df)

,row_count,rows_with_taxi,pct_with_taxi,rows_with_fhvhv,pct_with_fhvhv,rows_with_any_tlc,pct_with_any_tlc
0,1559590,1531505.0,98.2,1531505.0,98.2,1531505.0,98.2


Findings\. Nothing surprising here\. Bus and TLC provide the broadest coverage across every borough, while Subway coverage is concentrated in Manhattan and Brooklyn and drops off in Queens and Staten Island\. Traffic remains extremely sparse everywhere, which is exactly what we'd expect given the limited roadway sensor footprint\. The borough\-level view looks consistent with everything we learned during harmonization and doesn't reveal any merge issues\.

Now let's do a lightweight metric sanity check\. We are not looking for final analytical conclusions yet; we just want to catch impossible values like negative counts or obviously broken joins\.

In [17]:
# ---------------------------------------------------------------------
# Metric range sanity check for final Taxi/FHVHV fields
# ---------------------------------------------------------------------

tlc_metric_range_check_df = con.execute(f"""
    SELECT
        MIN(taxi_trip_count) AS min_taxi_trip_count,
        MAX(taxi_trip_count) AS max_taxi_trip_count,
        SUM(CASE WHEN taxi_trip_count < 0 THEN 1 ELSE 0 END) AS negative_taxi_trip_count_rows,
        MIN(fhvhv_trip_count) AS min_fhvhv_trip_count,
        MAX(fhvhv_trip_count) AS max_fhvhv_trip_count,
        SUM(CASE WHEN fhvhv_trip_count < 0 THEN 1 ELSE 0 END) AS negative_fhvhv_trip_count_rows,
        MAX(taxi_avg_trip_speed) AS max_taxi_avg_trip_speed,
        MAX(fhvhv_avg_trip_speed) AS max_fhvhv_avg_trip_speed,
        MAX(taxi_distinct_dropoff_zone_count) AS max_taxi_distinct_dropoff_zone_count,
        MAX(fhvhv_distinct_dropoff_zone_count) AS max_fhvhv_distinct_dropoff_zone_count
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

display(tlc_metric_range_check_df)

,min_taxi_trip_count,max_taxi_trip_count,negative_taxi_trip_count_rows,min_fhvhv_trip_count,max_fhvhv_trip_count,negative_fhvhv_trip_count_rows,max_taxi_avg_trip_speed,max_fhvhv_avg_trip_speed,max_taxi_distinct_dropoff_zone_count,max_fhvhv_distinct_dropoff_zone_count
0,0.0,3475.0,0.0,0.0,8655.0,0.0,100.0,66.037871,218,237


Findings\. The metric ranges all look reasonable and no negative values were detected across any modality\. Traffic volumes range from 0 to roughly 130K observations, bus speeds from about 2 to 38 mph, subway ridership from 1 to nearly 115K riders, and TLC activity from 1 to over 10K trips within a single mobility state\. The newly added Yellow, Green, and FHVHV trip counts also fall within plausible ranges, suggesting the service\-specific TLC metrics were integrated correctly\.

Finally, let's keep a small preview of the finished layer\. This is mostly for reviewer orientation: it shows the final shape of the panel without loading the full dataset into memory\.

In [18]:
merged_layer_preview_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{merged_mobility_layer_path}')
    USING SAMPLE 20 ROWS
""").df()

display(merged_layer_preview_df)

,taxi_zone_id,zone,borough,canonical_location_id,centroid_x,centroid_y,is_cbd_zone,is_cbd_gateway_zone,is_cbd_adjacent_zone,cbd_spatial_category,...,fhvhv_avg_trip_duration,fhvhv_avg_trip_speed,fhvhv_distinct_dropoff_zone_count,fhvhv_source_od_row_count,tlc_modality_count,tlc_contract_row_count,tlc_spatial_assignment_method,tlc_spatial_assignment_source,tlc_spatial_assignment_confidence,tlc_canonical_geography
0,4,Alphabet City,Manhattan,4,9.906340e+05,202959.782391,1.0,0.0,0.0,cbd,...,1127.023669,13.619405,99,272.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
1,4,Alphabet City,Manhattan,4,9.906340e+05,202959.782391,1.0,0.0,0.0,cbd,...,1281.960000,10.777372,113,233.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
2,4,Alphabet City,Manhattan,4,9.906340e+05,202959.782391,1.0,0.0,0.0,cbd,...,1093.095607,13.035061,86,187.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
3,5,Arden Heights,Staten Island,5,9.318714e+05,140681.351376,0.0,0.0,0.0,non_cbd,...,1278.833333,28.937000,11,12.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
4,6,Arrochar/Fort Wadsworth,Staten Island,6,9.643197e+05,157998.935612,0.0,0.0,0.0,non_cbd,...,940.366667,20.920478,24,41.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
5,18,Bedford Park,Bronx,18,1.014623e+06,255416.742153,0.0,0.0,0.0,non_cbd,...,1059.141026,12.927944,90,280.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
6,18,Bedford Park,Bronx,18,1.014623e+06,255416.742153,0.0,0.0,0.0,non_cbd,...,875.477941,16.645106,80,192.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
7,19,Bellerose,Queens,19,1.060003e+06,207352.352516,0.0,0.0,0.0,non_cbd,...,1015.482759,29.280668,42,62.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
8,19,Bellerose,Queens,19,1.060003e+06,207352.352516,0.0,0.0,0.0,non_cbd,...,988.174242,18.355035,35,64.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
9,19,Bellerose,Queens,19,1.060003e+06,207352.352516,0.0,0.0,0.0,non_cbd,...,1072.321586,20.079942,54,116.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone


Findings\. Everything looks normal here\. The random sample shows exactly what we'd expect from a dense multimodal panel: some mobility states contain observations from multiple systems, while others contain observations from only a subset of modalities\. 

⚠️ Follow\-Up Investigation\. Inspecting the merged output surfaced an important modeling question around the distinction between true zeros and missing observations\. Some fields may represent legitimate zero activity when observations are present, while others may use nulls to indicate a lack of coverage or source observations\. Before moving into exploratory analysis and feature engineering, we will review the harmonized source tables to determine how each mobility system represents zero activity versus non\-observation and establish a consistent missing\-value strategy for the merged mobility layer\.

### Section Summary

We combined Traffic, Bus, Subway, and TLC into a single dense mobility panel containing 3\.12 million Taxi Zone × Date × Temporal Bucket mobility states\. Along the way we validated the panel grain, resolved spatial edge cases introduced during harmonization, preserved modality\-specific missingness, and confirmed that all mobility measures merged successfully into a common analytical structure\. The resulting mobility layer is now ready for exploratory analysis, feature engineering, and downstream modeling\.

## 1\.3\.1\.6 Evaluating NaNs versus 0s

Before we start building models, let's take a closer look at what missing values actually mean across the different mobility systems\. Not every null represents the same thing: some may indicate that a modality was not observed in a particular mobility state, while others may represent situations where activity was known to be zero\. Understanding these differences now will help us establish a consistent missing\-value policy and avoid making incorrect assumptions during downstream analysis and feature engineering\.

Coverage alone doesn't tell us whether a missing value represents non\-observation or a true zero\. Let's compare nulls, zeros, and non\-zero observations for several key metrics to see whether each modality naturally distinguishes between zero activity and missing data\.

In [19]:
# ---------------------------------------------------------------------
# Missingness semantics for core analysis-ready mobility measures
# ---------------------------------------------------------------------

core_missingness_semantics_df = con.execute(f"""
    SELECT *
    FROM (
        SELECT 'traffic_volume' AS metric,
            SUM(CASE WHEN traffic_volume IS NULL THEN 1 ELSE 0 END) AS null_rows,
            SUM(CASE WHEN traffic_volume = 0 THEN 1 ELSE 0 END) AS zero_rows,
            SUM(CASE WHEN traffic_volume IS NOT NULL AND traffic_volume <> 0 THEN 1 ELSE 0 END) AS nonzero_rows
        FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
        UNION ALL
        SELECT 'bus_trip_count',
            SUM(CASE WHEN bus_trip_count IS NULL THEN 1 ELSE 0 END),
            SUM(CASE WHEN bus_trip_count = 0 THEN 1 ELSE 0 END),
            SUM(CASE WHEN bus_trip_count IS NOT NULL AND bus_trip_count <> 0 THEN 1 ELSE 0 END)
        FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
        UNION ALL
        SELECT 'subway_ridership',
            SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END),
            SUM(CASE WHEN subway_ridership = 0 THEN 1 ELSE 0 END),
            SUM(CASE WHEN subway_ridership IS NOT NULL AND subway_ridership <> 0 THEN 1 ELSE 0 END)
        FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
        UNION ALL
        SELECT 'taxi_trip_count',
            SUM(CASE WHEN taxi_trip_count IS NULL THEN 1 ELSE 0 END),
            SUM(CASE WHEN taxi_trip_count = 0 THEN 1 ELSE 0 END),
            SUM(CASE WHEN taxi_trip_count IS NOT NULL AND taxi_trip_count <> 0 THEN 1 ELSE 0 END)
        FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
        UNION ALL
        SELECT 'fhvhv_trip_count',
            SUM(CASE WHEN fhvhv_trip_count IS NULL THEN 1 ELSE 0 END),
            SUM(CASE WHEN fhvhv_trip_count = 0 THEN 1 ELSE 0 END),
            SUM(CASE WHEN fhvhv_trip_count IS NOT NULL AND fhvhv_trip_count <> 0 THEN 1 ELSE 0 END)
        FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
    )
""").df()

display(core_missingness_semantics_df)

,metric,null_rows,zero_rows,nonzero_rows
0,traffic_volume,1551170.0,97.0,8323.0
1,bus_trip_count,83112.0,0.0,1476478.0
2,subway_ridership,647377.0,0.0,912213.0
3,taxi_trip_count,28085.0,406367.0,1125138.0
4,fhvhv_trip_count,28085.0,8031.0,1523474.0


Findings\. This is interesting\. Traffic contains a small number of explicit zero\-volume observations, which suggests the source data distinguishes between zero activity and missing coverage\. Bus, Subway, and pooled TLC don't contain any zeros at all—rows either exist with activity or don't exist\. The TLC service\-specific fields tell a different story: Yellow, Green, and FHVHV all contain large numbers of explicit zeros, suggesting that zero\-trip states are already being represented separately from missing values\. Let's dig a little deeper before deciding how these remaining nulls should be handled\.

In [20]:
# ---------------------------------------------------------------------
# Zones with lowest TLC coverage
# ---------------------------------------------------------------------

low_tlc_coverage_zones_df = con.execute(f"""
    SELECT
        taxi_zone_id,
        ANY_VALUE(zone) AS zone,
        ANY_VALUE(borough) AS borough,
        SUM(CASE WHEN taxi_trip_count IS NULL AND fhvhv_trip_count IS NULL THEN 1 ELSE 0 END) AS null_tlc_rows,
        COUNT(*) AS panel_rows,
        ROUND(
            100.0 * SUM(CASE WHEN taxi_trip_count IS NULL AND fhvhv_trip_count IS NULL THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS pct_null_tlc_rows
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
    GROUP BY taxi_zone_id
    ORDER BY pct_null_tlc_rows DESC, taxi_zone_id
    LIMIT 20
""").df()

display(low_tlc_coverage_zones_df)

,taxi_zone_id,zone,borough,null_tlc_rows,panel_rows,pct_null_tlc_rows
0,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,5777.0,5930,97.42
1,199,Rikers Island,Bronx,5572.0,5930,93.96
2,110,Great Kills Park,Staten Island,4756.0,5930,80.20
3,2,Jamaica Bay,Queens,4508.0,5930,76.02
4,1,Newark Airport,EWR,4273.0,5930,72.06
5,253,Willets Point,Queens,846.0,5930,14.27
6,99,Freshkills Park,Staten Island,652.0,5930,10.99
7,8,Astoria Park,Queens,467.0,5930,7.88
8,27,Breezy Point/Fort Tilden/Riis Beach,Queens,371.0,5930,6.26
9,111,Green-Wood Cemetery,Brooklyn,342.0,5930,5.77


Findings\. This is encouraging\. The remaining TLC nulls are concentrated almost entirely in parks, islands, beaches, cemeteries, airports, and other special\-purpose locations where little or no trip activity would be expected\. After fixing the dense\-panel construction, TLC missingness dropped from more than 1\.5 million rows to just 25,015 rows\. At this point, the remaining TLC nulls look much more like legitimate zero\-trip mobility states than missing data coverage\.

### Subway NaN investigation

Let's take a closer look at Subway\. Unlike Traffic and TLC, the missing\-value behavior is still unclear\. Most subway\-served zones appear to have nearly complete coverage, but a handful of zones are missing hundreds or even thousands of expected mobility states\. Before deciding whether Subway nulls represent true missing observations or periods of zero ridership, we need to understand where these gaps occur and whether they reflect source\-data behavior, station coverage differences, or artifacts introduced during spatial aggregation\. Recall that there are 156 zones with subway coverage:

In [21]:
subway_zone_coverage_df = con.execute(f"""
    SELECT
        COUNT(DISTINCT taxi_zone_id) AS subway_zone_count
    FROM read_parquet('{subway_path}')
""").df()

display(subway_zone_coverage_df)

,subway_zone_count
0,156


In [22]:
subway_zone_gap_summary_df = con.execute(f"""
    WITH expected_states AS (
        SELECT
            taxi_zone_id,
            COUNT(*) AS expected_subway_states
        FROM dense_mobility_panel_base_df
        WHERE taxi_zone_id IN (
            SELECT DISTINCT taxi_zone_id
            FROM read_parquet('{subway_path}')
        )
        GROUP BY taxi_zone_id
    ),

    subway_zone_states AS (
        SELECT
            taxi_zone_id,
            COUNT(*) AS observed_subway_states
        FROM read_parquet('{subway_path}')
        GROUP BY taxi_zone_id
    )

    SELECT
        e.taxi_zone_id,
        z.zone,
        z.borough,
        e.expected_subway_states,
        COALESCE(s.observed_subway_states, 0) AS observed_subway_states,
        e.expected_subway_states - COALESCE(s.observed_subway_states, 0) AS missing_subway_states,
        ROUND(
            100.0 * COALESCE(s.observed_subway_states, 0) / e.expected_subway_states,
            2
        ) AS pct_expected_states_observed
    FROM expected_states e
    LEFT JOIN subway_zone_states s
        ON e.taxi_zone_id = s.taxi_zone_id
    LEFT JOIN (
        SELECT
            location_id AS taxi_zone_id,
            ANY_VALUE(zone) AS zone,
            ANY_VALUE(borough) AS borough
        FROM read_parquet('{taxi_zones_path}')
        GROUP BY location_id
    ) z
        ON e.taxi_zone_id = z.taxi_zone_id
    WHERE
        e.expected_subway_states - COALESCE(s.observed_subway_states, 0) > 100
    ORDER BY
        missing_subway_states DESC
""").df()

display(subway_zone_gap_summary_df)

,taxi_zone_id,zone,borough,expected_subway_states,observed_subway_states,missing_subway_states,pct_expected_states_observed
0,211,SoHo,Manhattan,5930,3272,2658,55.18
1,209,Seaport,Manhattan,5930,3655,2275,61.64
2,229,Sutton Place/Turtle Bay North,Manhattan,5930,3655,2275,61.64
3,68,East Chelsea,Manhattan,5930,5269,661,88.85
4,30,Broad Channel,Queens,5930,5278,652,89.01
5,201,Rockaway Park,Queens,5930,5280,650,89.04
6,86,Far Rockaway,Queens,5930,5359,571,90.37
7,117,Hammels/Arverne,Queens,5930,5419,511,91.38
8,178,Ocean Parkway South,Brooklyn,5930,5711,219,96.31
9,189,Prospect Heights,Brooklyn,5930,5763,167,97.18


Findings\. Several subway\-served zones are missing hundreds or even thousands of expected mobility states, including SoHo, Seaport, and Sutton Place/Turtle Bay North\. Because these areas are known to contain subway service, the missing observations cannot be explained simply by zones lacking subway stations\. Before establishing a final missing\-value policy, we need to determine whether these gaps reflect station closures, changes in station coverage, spatial assignment artifacts, periods of zero ridership that were dropped during aggregation, or genuine missing observations in the source data\.

Let's drill into a few of the zones with the largest Subway coverage gaps\. Rather than looking at aggregate counts, we'll inspect exactly which dates and temporal buckets are missing\. This should help us determine whether these gaps reflect isolated data issues, systematic coverage gaps, or mobility states that may have been dropped during aggregation\.

In [23]:
seaport_missing_runs_df = con.execute(f"""
    WITH expected_daily AS (
        SELECT
            CAST(date AS DATE) AS date,
            COUNT(*) AS expected_temporal_buckets
        FROM dense_mobility_panel_base_df
        WHERE taxi_zone_id = 209
        GROUP BY CAST(date AS DATE)
    ),

    observed_daily AS (
        SELECT
            CAST(date AS DATE) AS date,
            COUNT(*) AS observed_temporal_buckets,
            SUM(subway_ridership) AS subway_ridership
        FROM read_parquet('{subway_path}')
        WHERE taxi_zone_id = 209
        GROUP BY CAST(date AS DATE)
    ),

    daily_status AS (
        SELECT
            e.date,
            e.expected_temporal_buckets,
            COALESCE(o.observed_temporal_buckets, 0) AS observed_temporal_buckets,
            o.subway_ridership,
            CASE
                WHEN COALESCE(o.observed_temporal_buckets, 0) = 0
                THEN 1 ELSE 0
            END AS is_missing_day
        FROM expected_daily e
        LEFT JOIN observed_daily o
            ON e.date = o.date
    ),

    missing_days AS (
        SELECT
            date,
            DATE_DIFF('day', DATE '2023-01-01', date)
            - ROW_NUMBER() OVER (ORDER BY date) AS run_group
        FROM daily_status
        WHERE is_missing_day = 1
    )

    SELECT
        MIN(date) AS missing_start_date,
        MAX(date) AS missing_end_date,
        COUNT(*) AS missing_day_count
    FROM missing_days
    GROUP BY run_group
    ORDER BY missing_day_count DESC, missing_start_date
""").df()

display(seaport_missing_runs_df)

,missing_start_date,missing_end_date,missing_day_count
0,2025-01-01,2026-03-31,455


Findings\. This doesn't look random\. Entire zones disappear across all temporal buckets for days or weeks at a time before reappearing\. That feels much more like a service interruption, station closure, or other systematic event than ordinary missing data\. These nulls appear to carry real information and should be treated accordingly\.

Before deciding whether Subway nulls should remain null or be converted to zero, let's look at what happens immediately before and after one of the large missing periods\. If ridership disappears for a sustained period and then resumes at normal levels, that would be consistent with a temporary service disruption or closure\. If not, we'll need to revisit our assumptions\.

In [24]:
seaport_subway_timeline_df = con.execute(f"""
    SELECT
        date,
        temporal_bucket,
        subway_ridership,
        subway_transfers,
        subway_observation_count,
        subway_station_complex_count
    FROM read_parquet('{subway_path}')
    WHERE taxi_zone_id = 209
      AND date BETWEEN DATE '2024-12-15' AND DATE '2025-03-01'
    ORDER BY
        date,
        temporal_bucket
""").df()

display(seaport_subway_timeline_df)

,date,temporal_bucket,subway_ridership,subway_transfers,subway_observation_count,subway_station_complex_count
0,2024-12-15,weekend_am_peak,2713.0,7.0,33,1
1,2024-12-15,weekend_evening,3916.0,7.0,38,1
2,2024-12-15,weekend_midday,17388.0,62.0,59,1
3,2024-12-15,weekend_overnight,1603.0,2.0,47,1
4,2024-12-15,weekend_pm_peak,10953.0,32.0,40,1
...,...,...,...,...,...,...
80,2024-12-31,weekday_am_peak,5715.0,41.0,36,1
81,2024-12-31,weekday_evening,6050.0,2.0,38,1
82,2024-12-31,weekday_midday,18789.0,58.0,60,1
83,2024-12-31,weekday_overnight,1099.0,7.0,47,1


Findings\. Seaport shows healthy subway activity throughout 2024 and then disappears completely from the dataset beginning in 2025\. Combined with the earlier contiguous\-gap analysis, this doesn't look like random missing data\. Something happened at the station or service level, and the resulting nulls appear to carry real information\. Based on these findings, we'll treat Subway nulls within subway\-served zones as zero\-ridership states\. Zones that never had subway service will continue to remain null\.

### NaN Policy and Fix

Now that we understand what the nulls mean, let's apply the missing\-value policy directly to the merged layer\. Traffic and Bus nulls stay as nulls because they represent limited sensor or segment coverage\. TLC nulls become zero because missing TLC rows represent no observed trips\. Subway nulls become zero only for Taxi Zones that appear in the Subway dataset at least once; zones with no subway service remain null\.

In [25]:
fixed_merged_mobility_layer_path = MERGED_INTERMEDIATE_DIR / "merged_mobility_layer_fixed.parquet"

print("Input merged layer:", merged_mobility_layer_path)
print("Fixed merged layer:", fixed_merged_mobility_layer_path)

Input merged layer: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer.parquet
Fixed merged layer: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer_fixed.parquet


In [26]:
# ---------------------------------------------------------------------
# Archived fixed merged-layer rebuild
# ---------------------------------------------------------------------

fixed_merged_mobility_layer_path = merged_mobility_layer_path

print(
    "No separate fixed merged-layer rebuild is needed. "
    "1.2.4 now supplies final Taxi/FHVHV TLC metrics directly, "
    "and the current merged layer consumes those fields without downstream recomputation."
)

No separate fixed merged-layer rebuild is needed. 1.2.4 now supplies final Taxi/FHVHV TLC metrics directly, and the current merged layer consumes those fields without downstream recomputation.


Let's validate the fix\.

In [27]:
# ---------------------------------------------------------------------
# Current merged-layer null check for final TLC fields
# ---------------------------------------------------------------------

merged_tlc_null_check_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        SUM(CASE WHEN taxi_trip_count IS NULL THEN 1 ELSE 0 END) AS null_taxi_trip_count,
        SUM(CASE WHEN fhvhv_trip_count IS NULL THEN 1 ELSE 0 END) AS null_fhvhv_trip_count,
        SUM(CASE WHEN taxi_trip_count > 0 AND taxi_distinct_dropoff_zone_count IS NULL THEN 1 ELSE 0 END)
            AS active_taxi_rows_missing_distinct_dropoffs,
        SUM(CASE WHEN fhvhv_trip_count > 0 AND fhvhv_distinct_dropoff_zone_count IS NULL THEN 1 ELSE 0 END)
            AS active_fhvhv_rows_missing_distinct_dropoffs,
        SUM(CASE WHEN daypart IS NULL THEN 1 ELSE 0 END) AS null_daypart_rows,
        SUM(CASE WHEN temporal_bucket_order IS NULL THEN 1 ELSE 0 END) AS null_temporal_bucket_order_rows,
        SUM(CASE WHEN observation_sequence_id IS NULL THEN 1 ELSE 0 END) AS null_observation_sequence_rows
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

display(merged_tlc_null_check_df)

,row_count,null_taxi_trip_count,null_fhvhv_trip_count,active_taxi_rows_missing_distinct_dropoffs,active_fhvhv_rows_missing_distinct_dropoffs,null_daypart_rows,null_temporal_bucket_order_rows,null_observation_sequence_rows
0,1559590,28085.0,28085.0,0.0,0.0,0.0,0.0,0.0


Findings\. The fix worked\. TLC nulls are now gone, and the 25,015 formerly missing TLC states are now represented as zero\-trip states\. Subway now has 12,867 zero\-ridership states, while remaining Subway nulls are preserved for zones without subway coverage\. Traffic and Bus nulls were left untouched, which matches our policy\.

Before we close this out, let's make sure Subway now behaves cleanly at the zone level\. A Taxi Zone should either be subway\-served, with all Subway ridership values numeric, or non\-subway\-served, with all Subway ridership values null\. We should not have mixed zones where some Subway rows are numeric and others are still null\.

In [28]:
subway_zone_nan_policy_validation_df = con.execute(f"""
    SELECT
        taxi_zone_id,
        zone,
        borough,
        COUNT(*) AS panel_rows,
        SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END) AS null_subway_rows,
        SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) AS numeric_subway_rows,
        CASE
            WHEN SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END) = COUNT(*)
            THEN 'all_null'
            WHEN SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) = COUNT(*)
            THEN 'all_numeric'
            ELSE 'mixed'
        END AS subway_zone_status
    FROM read_parquet('{fixed_merged_mobility_layer_path}')
    GROUP BY
        taxi_zone_id,
        zone,
        borough
    HAVING subway_zone_status = 'mixed'
    ORDER BY
        taxi_zone_id
""").df()

display(subway_zone_nan_policy_validation_df)

,taxi_zone_id,zone,borough,panel_rows,null_subway_rows,numeric_subway_rows,subway_zone_status
0,3,Allerton/Pelham Gardens,Bronx,5930,66.0,5864.0,mixed
1,7,Astoria,Queens,5930,8.0,5922.0,mixed
2,14,Bay Ridge,Brooklyn,5930,8.0,5922.0,mixed
3,17,Bedford,Brooklyn,5930,8.0,5922.0,mixed
4,18,Bedford Park,Bronx,5930,8.0,5922.0,mixed
...,...,...,...,...,...,...,...
151,258,Woodhaven,Queens,5930,92.0,5838.0,mixed
152,259,Woodlawn/Wakefield,Bronx,5930,42.0,5888.0,mixed
153,260,Woodside,Queens,5930,8.0,5922.0,mixed
154,261,World Trade Center,Manhattan,5930,8.0,5922.0,mixed


Findings\. The validation passed\. No Taxi Zones exhibited mixed Subway behavior\. Every zone now falls cleanly into one of two categories: either Subway values are populated throughout the panel \(including zero\-ridership states\), or Subway remains entirely null because the zone has no subway coverage\. This confirms that the NaN policy was applied consistently and that the fixed mobility layer is internally coherent\.

### Bridge & Tunnel NaN vs Zero Check

Before closing the missing\-value policy work, let's do the same null\-versus\-zero check for Bridge & Tunnel\. This dataset stays facility\-level rather than Taxi Zone\-level, but we still need to understand whether missing values and zero volumes carry different meanings\.

In [29]:
bridge_tunnel_null_zero_df = con.execute(f"""
    SELECT
        'bridge_tunnel_volume' AS metric,
        COUNT(*) AS source_rows,
        SUM(CASE WHEN bridge_tunnel_volume IS NULL THEN 1 ELSE 0 END) AS null_rows,
        SUM(CASE WHEN bridge_tunnel_volume = 0 THEN 1 ELSE 0 END) AS zero_rows,
        SUM(CASE WHEN bridge_tunnel_volume IS NOT NULL AND bridge_tunnel_volume <> 0 THEN 1 ELSE 0 END) AS nonzero_rows,
        MIN(bridge_tunnel_volume) AS min_value,
        MAX(bridge_tunnel_volume) AS max_value
    FROM read_parquet('{bridge_tunnel_path}')
""").df()

display(bridge_tunnel_null_zero_df)

,metric,source_rows,null_rows,zero_rows,nonzero_rows,min_value,max_value
0,bridge_tunnel_volume,112657,0.0,0.0,112657.0,2.0,45695.0


Findings\. Bridge & Tunnel data contains no null values and no zero\-volume observations\. Every facility\-time period contains recorded traffic activity, indicating complete coverage across the study period and eliminating the need for any additional missing\-value handling\.

### Section Summary

This section focused on validating the final mobility panel and establishing a consistent missing\-value policy across all modalities\. We verified the merged layer's structure, coverage, geographic distribution, and metric ranges, then investigated the remaining null values in detail\. Traffic and Bus nulls were retained because they reflect limited observation coverage, while TLC nulls were converted to valid zero\-trip states\. Subway required additional investigation, which showed that missing observations were concentrated in a small number of subway\-served zones and occurred in long contiguous blocks rather than randomly\. Based on those findings, Subway nulls within subway\-served zones were converted to zero\-ridership states while zones without subway service retained null values\. The resulting mobility panel passed all validation checks and is now ready for downstream analysis and feature engineering\.

## 1\.3\.1\.7 Evaluate TLC Pooling Strategy

Adding service\-specific TLC metrics creates a modeling decision: should we analyze all TLC activity as a single pooled mobility signal, or should we preserve separate Yellow, Green, and FHVHV series? A pooled series is simpler and may provide a cleaner view of overall for\-hire mobility activity, but it could also mask meaningful differences between traditional taxis and app\-based services\. Before making that decision, we'll evaluate the relative contribution of each service, compare how the services behave over time, and examine whether they exhibit materially different geographic patterns\. Based on those findings, we'll decide whether to use a fully pooled TLC series, separate service\-specific series, or a hybrid approach that combines Yellow and Green while retaining FHVHV as a distinct mobility signal\.

### Validate TLC Service Reconciliation

Since TLC no longer carries a pooled final trip\-count field, this check now summarizes Taxi and FHVHV volumes directly\. The final design keeps the modalities separate while allowing downstream notebooks to combine them intentionally when a broader TLC activity signal is useful\.

In [30]:
# ---------------------------------------------------------------------
# Final TLC trip-volume summary
# ---------------------------------------------------------------------

tlc_trip_volume_summary_df = con.execute(f"""
    SELECT
        SUM(taxi_trip_count) AS total_taxi_trip_count,
        SUM(fhvhv_trip_count) AS total_fhvhv_trip_count,
        SUM(COALESCE(taxi_trip_count, 0) + COALESCE(fhvhv_trip_count, 0)) AS total_tlc_trip_count,
        SUM(taxi_source_od_row_count) AS total_taxi_source_od_rows,
        SUM(fhvhv_source_od_row_count) AS total_fhvhv_source_od_rows
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

display(tlc_trip_volume_summary_df)

,total_taxi_trip_count,total_fhvhv_trip_count,total_tlc_trip_count,total_taxi_source_od_rows,total_fhvhv_source_od_rows
0,139075726.0,778288736.0,917364462.0,53666126.0,272681609.0


Findings\. The final TLC panel now exposes Taxi and FHVHV as separate validated mobility signals\. Taxi already includes Yellow \+ Green from the upstream contract, while FHVHV remains its own modality\. This gives downstream notebooks cleaner choices without relying on a pooled TLC field that hides modality behavior\.

Let's also check the overall contribution of each service to the TLC ecosystem\.

In [31]:
# ---------------------------------------------------------------------
# Taxi/FHVHV volume by borough
# ---------------------------------------------------------------------

tlc_borough_volume_df = con.execute(f"""
    SELECT
        borough,
        SUM(taxi_trip_count) AS taxi_trips,
        SUM(fhvhv_trip_count) AS fhvhv_trips,
        ROUND(
            100.0 * SUM(taxi_trip_count)
            / NULLIF(SUM(SUM(taxi_trip_count)) OVER (), 0),
            2
        ) AS taxi_trip_share_pct,
        ROUND(
            100.0 * SUM(fhvhv_trip_count)
            / NULLIF(SUM(SUM(fhvhv_trip_count)) OVER (), 0),
            2
        ) AS fhvhv_trip_share_pct
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
    GROUP BY borough
    ORDER BY taxi_trips DESC NULLS LAST
""").df()

display(tlc_borough_volume_df)

,borough,taxi_trips,fhvhv_trips,taxi_trip_share_pct,fhvhv_trip_share_pct
0,Manhattan,121141294.0,299224144.0,87.10,38.45
1,Queens,13282700.0,164158412.0,9.55,21.09
2,Brooklyn,3357709.0,206060180.0,2.41,26.48
3,Bronx,702274.0,97174914.0,0.50,12.49
4,Unknown,580177.0,33807.0,0.42,0.00
5,Staten Island,9352.0,11637220.0,0.01,1.50
6,EWR,2220.0,59.0,0.00,0.00


Findings\. FHVHV dominates the TLC ecosystem, accounting for nearly 85% of all trips during the study period\. Yellow taxis contribute roughly 15% of trips, while Green taxis represent less than 1% of total activity\. Based on volume alone, Green appears too small to justify its own standalone mobility series, while the more meaningful modeling decision is whether Yellow and FHVHV should be analyzed together or separately\.

### Temporal Correlation

Before comparing the services visually, let's quantify how closely they move together over time\. If Yellow, Green, and FHVHV exhibit similar day\-to\-day patterns, then a pooled TLC series is likely capturing most of the underlying signal\. If the correlations are substantially lower, then separate series may contain additional information that would be lost through pooling\.

In [32]:
# ---------------------------------------------------------------------
# TLC field availability after upstream Taxi/FHVHV refactor
# ---------------------------------------------------------------------

expected_tlc_fields = [
    "taxi_trip_count",
    "taxi_avg_trip_distance",
    "taxi_avg_trip_duration",
    "taxi_avg_trip_speed",
    "taxi_distinct_dropoff_zone_count",
    "taxi_source_od_row_count",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_distance",
    "fhvhv_avg_trip_duration",
    "fhvhv_avg_trip_speed",
    "fhvhv_distinct_dropoff_zone_count",
    "fhvhv_source_od_row_count",
]

merged_columns_df = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

merged_column_names = set(merged_columns_df["column_name"])
missing_tlc_fields = sorted(set(expected_tlc_fields) - merged_column_names)

field_availability_df = pd.DataFrame({
    "field": expected_tlc_fields,
    "present": [field in merged_column_names for field in expected_tlc_fields],
})

display(field_availability_df)

if missing_tlc_fields:
    raise AssertionError(f"Merged mobility layer is missing expected TLC fields: {missing_tlc_fields}")

,field,present
0,taxi_trip_count,True
1,taxi_avg_trip_distance,True
2,taxi_avg_trip_duration,True
3,taxi_avg_trip_speed,True
4,taxi_distinct_dropoff_zone_count,True
5,taxi_source_od_row_count,True
6,fhvhv_trip_count,True
7,fhvhv_avg_trip_distance,True
8,fhvhv_avg_trip_duration,True
9,fhvhv_avg_trip_speed,True


Findings\. This argues against treating all TLC services as interchangeable\. The pooled TLC series is almost perfectly correlated with FHVHV, which makes sense because FHVHV accounts for most trips\. Yellow is only moderately correlated with FHVHV and the pooled series, while Green is basically moving on its own and contributes almost no volume\. This suggests the pooled TLC signal is mostly an FHVHV signal, not a balanced representation of all three services\.

Trip counts tell us whether demand moves together, but mobility systems can still behave differently even when volumes are correlated\. To determine whether Yellow and FHVHV exhibit similar transportation behavior, we'll compare the temporal correlation of trip distance, duration, and speed across the study period\.

In [33]:
# ---------------------------------------------------------------------
# Taxi/FHVHV average trip distance by temporal bucket
# ---------------------------------------------------------------------

tlc_distance_by_bucket_df = con.execute(f"""
    SELECT
        temporal_bucket,
        ANY_VALUE(temporal_bucket_order) AS temporal_bucket_order,
        SUM(taxi_avg_trip_distance * taxi_trip_count) / NULLIF(SUM(taxi_trip_count), 0) AS taxi_avg_trip_distance,
        SUM(fhvhv_avg_trip_distance * fhvhv_trip_count) / NULLIF(SUM(fhvhv_trip_count), 0) AS fhvhv_avg_trip_distance
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
    GROUP BY temporal_bucket
    ORDER BY temporal_bucket_order
""").df()

display(tlc_distance_by_bucket_df)

,temporal_bucket,temporal_bucket_order,taxi_avg_trip_distance,fhvhv_avg_trip_distance
0,weekday_overnight,1,4.932889,6.166675
1,weekday_am_peak,2,3.437024,4.798725
2,weekday_midday,3,3.327815,4.880443
3,weekday_pm_peak,4,2.987002,4.444413
4,weekday_evening,5,3.576544,5.341829
5,weekend_overnight,6,3.365753,5.445295
6,weekend_am_peak,7,4.358654,5.860346
7,weekend_midday,8,3.434174,4.876283
8,weekend_pm_peak,9,3.444590,4.751563
9,weekend_evening,10,3.819019,4.973197


In [34]:
# ---------------------------------------------------------------------
# Taxi/FHVHV average trip duration by temporal bucket
# ---------------------------------------------------------------------

tlc_duration_by_bucket_df = con.execute(f"""
    SELECT
        temporal_bucket,
        ANY_VALUE(temporal_bucket_order) AS temporal_bucket_order,
        SUM(taxi_avg_trip_duration * taxi_trip_count) / NULLIF(SUM(taxi_trip_count), 0) AS taxi_avg_trip_duration,
        SUM(fhvhv_avg_trip_duration * fhvhv_trip_count) / NULLIF(SUM(fhvhv_trip_count), 0) AS fhvhv_avg_trip_duration
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
    GROUP BY temporal_bucket
    ORDER BY temporal_bucket_order
""").df()

display(tlc_duration_by_bucket_df)

,temporal_bucket,temporal_bucket_order,taxi_avg_trip_duration,fhvhv_avg_trip_duration
0,weekday_overnight,1,946.646269,1037.849498
1,weekday_am_peak,2,1076.559716,1189.420893
2,weekday_midday,3,1201.741845,1322.578270
3,weekday_pm_peak,4,1101.021614,1275.226413
4,weekday_evening,5,942.136032,1120.818634
5,weekend_overnight,6,882.097365,1053.951053
6,weekend_am_peak,7,869.991877,1027.170264
7,weekend_midday,8,1023.780212,1180.909925
8,weekend_pm_peak,9,1099.437624,1239.144309
9,weekend_evening,10,1008.827026,1126.276954


Confirm that the rebuilt final panels preserve their established grain while carrying the new static Taxi Zone context into every physical mobility row\.

### Behavioral Comparison

In [35]:
# ---------------------------------------------------------------------
# Taxi/FHVHV weighted metric summary
# ---------------------------------------------------------------------

tlc_weighted_metric_summary_df = con.execute(f"""
    SELECT
        'taxi' AS modality,
        SUM(taxi_trip_count) AS trip_count,
        SUM(taxi_avg_trip_distance * taxi_trip_count) / NULLIF(SUM(taxi_trip_count), 0) AS avg_trip_distance,
        SUM(taxi_avg_trip_duration * taxi_trip_count) / NULLIF(SUM(taxi_trip_count), 0) AS avg_trip_duration,
        SUM(taxi_avg_trip_speed * taxi_trip_count) / NULLIF(SUM(taxi_trip_count), 0) AS avg_trip_speed
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')

    UNION ALL

    SELECT
        'fhvhv' AS modality,
        SUM(fhvhv_trip_count) AS trip_count,
        SUM(fhvhv_avg_trip_distance * fhvhv_trip_count) / NULLIF(SUM(fhvhv_trip_count), 0) AS avg_trip_distance,
        SUM(fhvhv_avg_trip_duration * fhvhv_trip_count) / NULLIF(SUM(fhvhv_trip_count), 0) AS avg_trip_duration,
        SUM(fhvhv_avg_trip_speed * fhvhv_trip_count) / NULLIF(SUM(fhvhv_trip_count), 0) AS avg_trip_speed
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

display(tlc_weighted_metric_summary_df)

,modality,trip_count,avg_trip_distance,avg_trip_duration,avg_trip_speed
0,taxi,139075726.0,3.435325,1063.924218,10.941656
1,fhvhv,778288736.0,5.018810,1192.509548,15.187035


Findings\. Taxi and FHVHV are clearly not interchangeable just because both live inside the TLC family\. Taxi carries far fewer trips overall, while the weighted distance, duration, and speed summaries show a meaningfully different behavioral profile from FHVHV rather than a trivial rescaling of the same signal\. That is exactly why the upstream refactor was worth doing: the merged layer now arrives with two cleaner TLC modalities that can be interpreted separately downstream\.

### Geographic Check

Temporal and behavioral differences tell us whether the services move differently over time, but they do not tell us whether they serve different parts of the city\. To complete the pooling assessment, we'll compare the geographic footprint of Yellow, Green, and FHVHV trips\. If the services exhibit similar spatial distributions, pooling becomes easier to justify\. If they are concentrated in different boroughs, then separate series may preserve meaningful geographic information that would otherwise be lost\.

In [36]:
# ---------------------------------------------------------------------
# Taxi/FHVHV volume by temporal bucket
# ---------------------------------------------------------------------

tlc_temporal_bucket_volume_df = con.execute(f"""
    SELECT
        temporal_bucket,
        ANY_VALUE(temporal_bucket_order) AS temporal_bucket_order,
        SUM(taxi_trip_count) AS taxi_trip_count,
        SUM(fhvhv_trip_count) AS fhvhv_trip_count,
        ROUND(
            100.0 * SUM(taxi_trip_count)
            / NULLIF(SUM(SUM(taxi_trip_count)) OVER (), 0),
            2
        ) AS taxi_trip_share_pct,
        ROUND(
            100.0 * SUM(fhvhv_trip_count)
            / NULLIF(SUM(SUM(fhvhv_trip_count)) OVER (), 0),
            2
        ) AS fhvhv_trip_share_pct
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
    GROUP BY temporal_bucket
    ORDER BY temporal_bucket_order
""").df()

display(tlc_temporal_bucket_volume_df)

,temporal_bucket,temporal_bucket_order,taxi_trip_count,fhvhv_trip_count,taxi_trip_share_pct,fhvhv_trip_share_pct
0,weekday_overnight,1,4924282.0,53012796.0,3.54,6.81
1,weekday_am_peak,2,14339843.0,103264465.0,10.31,13.27
2,weekday_midday,3,31850693.0,145407979.0,22.90,18.68
3,weekday_pm_peak,4,27231666.0,122944553.0,19.58,15.80
4,weekday_evening,5,21817596.0,111976622.0,15.69,14.39
5,weekend_overnight,6,6732874.0,47277925.0,4.84,6.07
6,weekend_am_peak,7,2829205.0,25975334.0,2.03,3.34
7,weekend_midday,8,12194956.0,66344977.0,8.77,8.52
8,weekend_pm_peak,9,9504960.0,51736570.0,6.83,6.65
9,weekend_evening,10,7649651.0,50347515.0,5.50,6.47


Findings\. The temporal\-bucket volume split reinforces that Taxi and FHVHV behave like related but distinct modalities\. Both services still show recognizable time\-of\-week structure, but their bucket shares are not identical, which gives the merged layer more signal than a single pooled TLC series would have\. Keeping Taxi and FHVHV separate preserves those timing differences for later EDA, anomaly work, and supervised modeling\.

### Section Summary and Recommendation

The results point to a pretty clear answer: 

- Green contributes almost no activity and doesn't appear large enough to justify its own mobility series\. At the other extreme, the pooled TLC metrics are largely an FHVHV signal because FHVHV accounts for roughly 85% of all trips\. 

- Yellow and FHVHV, however, consistently behaved differently throughout the analysis\. They exhibited only moderate trip\-count correlation, very different distance and duration patterns, and noticeably different geographic footprints\. 

- Yellow activity remains heavily concentrated in Manhattan, while FHVHV activity is much more broadly distributed across the city\. 

Based on these findings, we'll combine Yellow and Green into a single Taxi mobility series and retain FHVHV as a separate modality\. This preserves the most meaningful behavioral differences while avoiding unnecessary feature duplication and complexity in the final mobility panel\.

## 1\.3\.1\.8 Write Final Outputs

At this point the merged mobility panel design is already resolved upstream, so this final step is mostly about packaging\. We write the audit\-only and analysis\-ready mobility panels, then carry Bridge & Tunnel, weather, and the enriched Taxi Zone spatial/context tables forward as separate companion outputs\. That keeps the legacy mobility panels stable for downstream notebooks while still making the contextual layers available for app work, interpretation, and later supervised feature use\.

### Reconstruct Taxi Destination Diversity

Taxi destination diversity is no longer reconstructed in 1\.3\.1\. That work now happens upstream in 1\.2\.3 while the Pickup Zone × Dropoff Zone detail is still available, and 1\.2\.4 carries the validated result forward as taxi\_distinct\_dropoff\_zone\_count\. Keeping this section as an explicit archive note preserves the storyline without reintroducing the old downstream estimation/reconstruction path\.

In [37]:
# ---------------------------------------------------------------------
# Archived downstream Taxi destination-diversity reconstruction
# ---------------------------------------------------------------------

print(
    "Skipping downstream Taxi distinct-dropoff reconstruction. "
    "taxi_distinct_dropoff_zone_count is computed upstream in 1.2.3 "
    "and carried through the 1.2.4 harmonized TLC table."
)

taxi_distinct_dropoff_df = con.execute(f"""
    SELECT
        taxi_zone_id,
        date,
        temporal_bucket,
        taxi_distinct_dropoff_zone_count
    FROM read_parquet('{tlc_path.as_posix()}')
    WHERE taxi_distinct_dropoff_zone_count IS NOT NULL
    LIMIT 10
""").df()

display(taxi_distinct_dropoff_df)

Skipping downstream Taxi distinct-dropoff reconstruction. taxi_distinct_dropoff_zone_count is computed upstream in 1.2.3 and carried through the 1.2.4 harmonized TLC table.


,taxi_zone_id,date,temporal_bucket,taxi_distinct_dropoff_zone_count
0,1,2023-01-01,weekend_midday,2
1,1,2023-01-01,weekend_pm_peak,3
2,1,2023-01-02,weekday_am_peak,1
3,1,2023-01-02,weekday_midday,2
4,1,2023-01-03,weekday_midday,1
5,1,2023-01-05,weekday_am_peak,1
6,1,2023-01-05,weekday_midday,3
7,1,2023-01-06,weekday_pm_peak,1
8,1,2023-01-06,weekday_evening,1
9,1,2023-01-08,weekend_pm_peak,1


In [38]:
# ---------------------------------------------------------------------
# Validate upstream Taxi destination-diversity availability
# ---------------------------------------------------------------------

taxi_distinct_dropoff_summary_df = con.execute(f"""
    SELECT
        COUNT(*) AS tlc_rows,
        SUM(CASE WHEN taxi_trip_count > 0 THEN 1 ELSE 0 END) AS active_taxi_rows,
        SUM(
            CASE
                WHEN taxi_trip_count > 0 AND taxi_distinct_dropoff_zone_count IS NULL
                THEN 1 ELSE 0
            END
        ) AS active_taxi_rows_missing_distinct_dropoffs,
        MIN(taxi_distinct_dropoff_zone_count) AS min_taxi_distinct_dropoff_zone_count,
        MAX(taxi_distinct_dropoff_zone_count) AS max_taxi_distinct_dropoff_zone_count
    FROM read_parquet('{tlc_path.as_posix()}')
""").df()

display(taxi_distinct_dropoff_summary_df)

if taxi_distinct_dropoff_summary_df.loc[0, "active_taxi_rows_missing_distinct_dropoffs"] != 0:
    raise AssertionError(
        "Active Taxi rows are missing taxi_distinct_dropoff_zone_count in the 1.2.4 TLC table."
    )

,tlc_rows,active_taxi_rows,active_taxi_rows_missing_distinct_dropoffs,min_taxi_distinct_dropoff_zone_count,max_taxi_distinct_dropoff_zone_count
0,1531505,1125138.0,0.0,1,218


Findings\. Taxi destination diversity is now available from the upstream TLC contract rather than being rebuilt here\. That is the cleaner design: the distinct dropoff count is computed before Yellow and Green are collapsed into Taxi, so we avoid double\-counting overlap without forcing 1\.3\.1 to scan raw TLC records again\.

### Create Audit\-Only Mobility Panel

The Audit\-Only Mobility Panel applies the final TLC representation while preserving supporting QA, coverage, variability, and diagnostic fields\. This dataset serves as the project's authoritative validation layer and can be used to investigate anomalies, verify transformations, and troubleshoot future issues without requiring regeneration of intermediate datasets\.

In [39]:
# ---------------------------------------------------------------------
# Refresh merged-layer spatial context from 1.2.6 source-of-truth
# ---------------------------------------------------------------------

if not merged_mobility_layer_path.exists():
    raise FileNotFoundError(
        "merged_mobility_layer.parquet does not exist yet. Build the merged layer before refreshing spatial context."
    )

refreshed_merged_mobility_layer_path = (
    MERGED_INTERMEDIATE_DIR / "merged_mobility_layer_spatial_refresh.parquet"
)

con.execute(f"""
    COPY (
        WITH latest_spatial_context AS (
            SELECT
                location_id AS taxi_zone_id,
                zone,
                borough,
                canonical_location_id,
                centroid_x,
                centroid_y,
                is_cbd_zone,
                is_cbd_gateway_zone,
                is_cbd_adjacent_zone,
                cbd_spatial_category,
                distance_to_cbd_feet,
                distance_to_cbd_miles,
                distance_to_gateway_feet,
                distance_to_gateway_miles,
                connected_neighbor_count,
                network_betweenness_centrality,
                network_closeness_centrality,
                network_clustering_coefficient
            FROM read_parquet('{taxi_zones_path.as_posix()}')
        )

        SELECT
            merged.taxi_zone_id,
            spatial.zone,
            spatial.borough,
            spatial.canonical_location_id,
            spatial.centroid_x,
            spatial.centroid_y,
            spatial.is_cbd_zone,
            spatial.is_cbd_gateway_zone,
            spatial.is_cbd_adjacent_zone,
            spatial.cbd_spatial_category,
            spatial.distance_to_cbd_feet,
            spatial.distance_to_cbd_miles,
            spatial.distance_to_gateway_feet,
            spatial.distance_to_gateway_miles,
            spatial.connected_neighbor_count,
            spatial.network_betweenness_centrality,
            spatial.network_closeness_centrality,
            spatial.network_clustering_coefficient,
            merged.date,
            merged.year,
            merged.month,
            merged.day_of_week,
            merged.daypart,
            merged.daypart_order,
            merged.temporal_bucket,
            merged.temporal_bucket_order,
            merged.observation_sequence_id,
            merged.pre_post_cp,
            merged.traffic_volume,
            merged.traffic_observation_count,
            merged.traffic_distinct_segment_count,
            merged.avg_bus_speed,
            merged.avg_bus_travel_time,
            merged.bus_speed_stddev,
            merged.bus_travel_time_stddev,
            merged.bus_trip_count,
            merged.bus_segment_observation_count,
            merged.bus_assignment_method_count,
            merged.subway_ridership,
            merged.subway_transfers,
            merged.subway_observation_count,
            merged.subway_station_complex_count,
            merged.taxi_trip_count,
            merged.taxi_avg_trip_distance,
            merged.taxi_avg_trip_duration,
            merged.taxi_avg_trip_speed,
            merged.taxi_distinct_dropoff_zone_count,
            merged.taxi_source_od_row_count,
            merged.fhvhv_trip_count,
            merged.fhvhv_avg_trip_distance,
            merged.fhvhv_avg_trip_duration,
            merged.fhvhv_avg_trip_speed,
            merged.fhvhv_distinct_dropoff_zone_count,
            merged.fhvhv_source_od_row_count,
            merged.tlc_modality_count,
            merged.tlc_contract_row_count,
            merged.tlc_spatial_assignment_method,
            merged.tlc_spatial_assignment_source,
            merged.tlc_spatial_assignment_confidence,
            merged.tlc_canonical_geography
        FROM read_parquet('{merged_mobility_layer_path.as_posix()}') AS merged
        LEFT JOIN latest_spatial_context AS spatial
            USING (taxi_zone_id)
    )
    TO '{refreshed_merged_mobility_layer_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

shutil.move(
    refreshed_merged_mobility_layer_path.as_posix(),
    merged_mobility_layer_path.as_posix(),
)

print(f"Refreshed merged-layer spatial context: {merged_mobility_layer_path}")

Refreshed merged-layer spatial context: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer.parquet


In [40]:
# ---------------------------------------------------------------------
# Create audit-only mobility panel
# ---------------------------------------------------------------------

audit_only_mobility_panel_df = con.execute(f"""
    SELECT
        taxi_zone_id, zone, borough, canonical_location_id,
        centroid_x, centroid_y,
        is_cbd_zone, is_cbd_gateway_zone, is_cbd_adjacent_zone,
        cbd_spatial_category,
        distance_to_cbd_feet, distance_to_cbd_miles,
        distance_to_gateway_feet, distance_to_gateway_miles,
        connected_neighbor_count,
        network_betweenness_centrality,
        network_closeness_centrality,
        network_clustering_coefficient,
        date, year, month, day_of_week, daypart, daypart_order,
        temporal_bucket, temporal_bucket_order, observation_sequence_id, pre_post_cp,
        traffic_volume, traffic_observation_count, traffic_distinct_segment_count,
        avg_bus_speed, avg_bus_travel_time, bus_speed_stddev, bus_travel_time_stddev,
        bus_trip_count, bus_segment_observation_count, bus_assignment_method_count,
        subway_ridership, subway_transfers, subway_observation_count, subway_station_complex_count,
        taxi_trip_count, taxi_avg_trip_distance, taxi_avg_trip_duration, taxi_avg_trip_speed,
        taxi_distinct_dropoff_zone_count, taxi_source_od_row_count,
        fhvhv_trip_count, fhvhv_avg_trip_distance, fhvhv_avg_trip_duration, fhvhv_avg_trip_speed,
        fhvhv_distinct_dropoff_zone_count, fhvhv_source_od_row_count,
        tlc_modality_count, tlc_contract_row_count,
        tlc_spatial_assignment_method, tlc_spatial_assignment_source,
        tlc_spatial_assignment_confidence, tlc_canonical_geography
    FROM read_parquet('{merged_mobility_layer_path.as_posix()}')
""").df()

display(audit_only_mobility_panel_df.head())
print(f"Audit-only mobility panel rows: {len(audit_only_mobility_panel_df):,}")

,taxi_zone_id,zone,borough,canonical_location_id,centroid_x,centroid_y,is_cbd_zone,is_cbd_gateway_zone,is_cbd_adjacent_zone,cbd_spatial_category,...,fhvhv_avg_trip_duration,fhvhv_avg_trip_speed,fhvhv_distinct_dropoff_zone_count,fhvhv_source_od_row_count,tlc_modality_count,tlc_contract_row_count,tlc_spatial_assignment_method,tlc_spatial_assignment_source,tlc_spatial_assignment_confidence,tlc_canonical_geography
0,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,1109.781553,22.520018,97,221.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
1,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,978.073171,25.179681,57,84.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
2,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,1092.743842,20.465489,76,127.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
3,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,1152.000000,19.206710,54,85.0,1,1,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone
4,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,1014.854015,24.359029,64,96.0,2,2,native_tlc_taxi_zone_id,tlc_source_pickup_location_id,native,taxi_zone


Audit-only mobility panel rows: 1,559,590


### Create Analysis\-Ready Mobility Panel

With the audit layer preserved, we can create a streamlined analytical dataset focused on mobility signals rather than data quality diagnostics\. This version removes supporting QA and coverage fields, leaving a cleaner set of features for feature engineering, dimensionality reduction, anomaly detection, and congestion\-pricing analysis\.

Weather is now available as a finalized Taxi Zone contextual layer from \`1\.2\.5\`, but we are intentionally not merging it into the legacy mobility panels here\. To preserve downstream compatibility, \`1\.3\.1\` carries weather forward as its own companion final table, just like Bridge & Tunnel and the Taxi Zone spatial reference\.

In [41]:
# Keep the legacy analysis-ready panel stable for downstream consumers.
# Weather, Bridge & Tunnel, and other context layers stay separate companion tables.
analysis_ready_mobility_panel_df = con.execute("""
    SELECT
        taxi_zone_id, zone, borough, canonical_location_id,
        centroid_x, centroid_y,
        is_cbd_zone, is_cbd_gateway_zone, is_cbd_adjacent_zone,
        cbd_spatial_category,
        distance_to_cbd_miles, distance_to_gateway_miles,
        connected_neighbor_count,
        network_betweenness_centrality,
        network_closeness_centrality,
        network_clustering_coefficient,
        date, year, month, day_of_week, daypart, daypart_order,
        temporal_bucket, temporal_bucket_order, observation_sequence_id, pre_post_cp,
        traffic_volume,
        bus_trip_count, avg_bus_speed, avg_bus_travel_time,
        subway_ridership, subway_transfers,
        taxi_trip_count, taxi_avg_trip_distance, taxi_avg_trip_duration,
        taxi_avg_trip_speed, taxi_distinct_dropoff_zone_count,
        fhvhv_trip_count, fhvhv_avg_trip_distance, fhvhv_avg_trip_duration,
        fhvhv_avg_trip_speed, fhvhv_distinct_dropoff_zone_count
    FROM audit_only_mobility_panel_df
""").df()

display(analysis_ready_mobility_panel_df.head())
print(f"Analysis-ready mobility panel rows: {len(analysis_ready_mobility_panel_df):,}")

,taxi_zone_id,zone,borough,canonical_location_id,centroid_x,centroid_y,is_cbd_zone,is_cbd_gateway_zone,is_cbd_adjacent_zone,cbd_spatial_category,...,taxi_trip_count,taxi_avg_trip_distance,taxi_avg_trip_duration,taxi_avg_trip_speed,taxi_distinct_dropoff_zone_count,fhvhv_trip_count,fhvhv_avg_trip_distance,fhvhv_avg_trip_duration,fhvhv_avg_trip_speed,fhvhv_distinct_dropoff_zone_count
0,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,1.0,5.07,749.0,24.368491,1,412.0,6.942306,1109.781553,22.520018,97
1,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,0.0,NaN,NaN,NaN,<NA>,123.0,6.840992,978.073171,25.179681,57
2,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,0.0,NaN,NaN,NaN,<NA>,203.0,6.212094,1092.743842,20.465489,76
3,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,0.0,NaN,NaN,NaN,<NA>,136.0,6.146147,1152.000000,19.206710,54
4,53,College Point,Queens,53,1.027782e+06,223823.082735,0.0,0.0,0.0,non_cbd,...,1.0,5.56,904.0,22.141593,1,137.0,6.866905,1014.854015,24.359029,64


Analysis-ready mobility panel rows: 1,559,590


### Write to File

In [42]:
audit_only_output_path = MERGED_FINAL_DIR / "audit_only_mobility_panel.parquet"
analysis_ready_output_path = MERGED_FINAL_DIR / "analysis_ready_mobility_panel.parquet"

audit_only_mobility_panel_df.to_parquet(
    audit_only_output_path,
    index=False
)

analysis_ready_mobility_panel_df.to_parquet(
    analysis_ready_output_path,
    index=False
)

print(f"Audit output: {audit_only_output_path}")
print(f"Analysis output: {analysis_ready_output_path}")

Audit output: pipeline_data/1.3.1.final_tables/audit_only_mobility_panel.parquet
Analysis output: pipeline_data/1.3.1.final_tables/analysis_ready_mobility_panel.parquet


In [43]:
# ---------------------------------------------------------------------
# Validate written analysis-ready spatial-context carry-through
# ---------------------------------------------------------------------

if not analysis_ready_output_path.exists():
    raise FileNotFoundError(
        "analysis_ready_mobility_panel.parquet does not exist yet. Run the write block before validation."
    )

final_spatial_context_validation_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS distinct_analysis_keys,
        COUNT(*) - COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS duplicate_key_rows,
        SUM(CASE WHEN zone IS NOT NULL AND cbd_spatial_category IS NULL THEN 1 ELSE 0 END)
            AS physical_rows_missing_spatial_category,
        SUM(CASE WHEN zone IS NOT NULL AND distance_to_cbd_miles IS NULL THEN 1 ELSE 0 END)
            AS physical_rows_missing_cbd_distance,
        SUM(CASE WHEN zone IS NOT NULL AND connected_neighbor_count IS NULL THEN 1 ELSE 0 END)
            AS physical_rows_missing_connected_neighbor_count,
        SUM(CASE WHEN zone IS NOT NULL AND network_betweenness_centrality IS NULL THEN 1 ELSE 0 END)
            AS physical_rows_missing_network_betweenness,
        SUM(CASE WHEN zone IS NOT NULL AND network_closeness_centrality IS NULL THEN 1 ELSE 0 END)
            AS physical_rows_missing_network_closeness,
        SUM(CASE WHEN zone IS NOT NULL AND network_clustering_coefficient IS NULL THEN 1 ELSE 0 END)
            AS physical_rows_missing_network_clustering,
        COUNT(DISTINCT cbd_spatial_category) AS distinct_spatial_categories
    FROM read_parquet('{analysis_ready_output_path.as_posix()}')
""").df()

display(final_spatial_context_validation_df)

,row_count,distinct_analysis_keys,duplicate_key_rows,physical_rows_missing_spatial_category,physical_rows_missing_cbd_distance,physical_rows_missing_connected_neighbor_count,physical_rows_missing_network_betweenness,physical_rows_missing_network_closeness,physical_rows_missing_network_clustering,distinct_spatial_categories
0,1559590,1559590,0,11860.0,11860.0,11860.0,11860.0,11860.0,11860.0,4


Findings\. The final spatial\-context validation confirms that the enriched zone fields made it all the way into the analysis\-ready panel without breaking the panel grain\. The Taxi Zone x date x temporal bucket key remains unique, the physical zones retain their CBD classification and distance fields, and the new graph\-based connectivity fields are still available downstream\. In short, the merged panel is carrying forward the 1\.2\.6 spatial features the way we intended rather than dropping them during packaging\.

Copy the Bridge & Tunnel file forward

In [44]:
bridge_tunnel_output_path = MERGED_FINAL_DIR / "bridge_tunnel_mobility_panel.parquet"

con.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{bridge_tunnel_path}')
    )
    TO '{bridge_tunnel_output_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Bridge & Tunnel output: {bridge_tunnel_output_path}")

Bridge & Tunnel output: pipeline_data/1.3.1.final_tables/bridge_tunnel_mobility_panel.parquet


Carry the finalized weather context table forward as a companion output as well\. Like Bridge & Tunnel, this remains a separate contextual dataset rather than being merged into the legacy mobility\-only panels\.

In [45]:
weather_context_output_path = MERGED_FINAL_DIR / "weather_taxi_zone_context.parquet"

con.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{weather_context_path}')
    )
    TO '{weather_context_output_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Weather context output: {weather_context_output_path}")

Weather context output: pipeline_data/1.3.1.final_tables/weather_taxi_zone_context.parquet


Carry the enriched Taxi Zone spatial reference and connectivity table forward for downstream analysis and app use\.

In [46]:
# ---------------------------------------------------------------------
# Carry forward enriched Taxi Zone spatial context and connectivity
# ---------------------------------------------------------------------

con.execute(f"""
    COPY (SELECT * FROM read_parquet('{taxi_zones_path.as_posix()}'))
    TO '{taxi_zones_output_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
    COPY (SELECT * FROM read_parquet('{taxi_zone_connectivity_path.as_posix()}'))
    TO '{taxi_zone_connectivity_output_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Taxi Zone spatial context: {taxi_zones_output_path}")
print(f"Taxi Zone connectivity: {taxi_zone_connectivity_output_path}")

Taxi Zone spatial context: pipeline_data/1.3.1.final_tables/nyc_taxi_zones_harmonized.parquet
Taxi Zone connectivity: pipeline_data/1.3.1.final_tables/taxi_zone_connectivity.parquet


Final output size check

In [47]:
final_output_size_df = pd.DataFrame(
    {
        "dataset": [
            "Audit Only",
            "Analysis Ready",
        ],
        "rows": [
            con.execute(f"SELECT COUNT(*) FROM read_parquet('{audit_only_output_path.as_posix()}')").fetchone()[0],
            con.execute(f"SELECT COUNT(*) FROM read_parquet('{analysis_ready_output_path.as_posix()}')").fetchone()[0],
        ],
        "columns": [
            len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{audit_only_output_path.as_posix()}')").df()),
            len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{analysis_ready_output_path.as_posix()}')").df()),
        ],
    }
)

display(final_output_size_df)

,dataset,rows,columns
0,Audit Only,1559590,60
1,Analysis Ready,1559590,42


## Close

This notebook built the dense Taxi Zone x Date x Temporal Bucket mobility layer that downstream analysis now depends on\. We attached the enriched Taxi Zone spatial context from \`1\.2\.6\`, integrated the harmonized Traffic, Bus, Subway, and final Taxi/FHVHV TLC measures, preserved modality\-specific missingness in both audit and analysis\-ready panels, and then carried Bridge & Tunnel and weather forward as separate companion tables rather than forcing them into the legacy mobility\-only panel contract\. The result is a cleaner merged mobility foundation: one stable multimodal panel for core analysis, plus separate contextual layers that downstream notebooks and the app can join when they actually need them\.

### Outputs for Downstream Analysis

The following datasets were generated in this notebook:

Analysis\-Ready Mobility Panel \(Primary Dataset\)

- pipeline\_data/1\.3\.1\.final\_tables/analysis\_ready\_mobility\_panel\.parquet

- Use for: EDA, feature engineering, dimensionality reduction, anomaly detection, congestion\-pricing analysis, and all downstream modeling activities\.

Audit\-Only Mobility Panel \(Validation & Troubleshooting\)

- pipeline\_data/1\.3\.1\.final\_tables/audit\_only\_mobility\_panel\.parquet

- Use for: QA, validation, troubleshooting, missing\-data investigations, metric reconciliation, and methodological review\.

Bridge & Tunnel Mobility Dataset \(Facility\-Level Context\)

- pipeline\_data/1\.3\.1\.final\_tables/bridge\_tunnel\_mobility\_panel\.parquet

- Use for: contextual mobility analysis, system\-level validation, and potential anomaly corroboration\. This dataset remains separate from the Taxi Zone mobility panel\.

Weather Context Dataset \(Taxi Zone Contextual Companion\)

- pipeline\_data/1\.3\.1\.final\_tables/weather\_taxi\_zone\_context\.parquet

- Use for: exploratory analysis, anomaly interpretation, supervised learning features, forecasting covariates, and app\-side contextual overlays\. This dataset remains separate from the legacy mobility\-only panels so downstream notebooks can opt into weather without breaking existing joins\.

Taxi Zone Reference Layer \(Canonical Geography\)

- pipeline\_data/1\.3\.1\.final\_tables/nyc\_taxi\_zones\_harmonized\.parquet

- Use for: spatial feature engineering, geospatial visualization, neighborhood spillover analysis, borough\-level aggregation, congestion\-pricing geography classification, CBD proximity calculations, and all downstream spatial joins\. This dataset serves as the project's canonical geographic reference layer and should be used whenever Taxi Zone attributes, boundaries, or spatial relationships are required\.

### Looking Ahead

The next notebooks \(1\.4\.x Exploratory Data Analysis and 1\.5x Feature Engineering\) should primarily load:

- pipeline\_data/1\.3\.1\.final\_tables/analysis\_ready\_mobility\_panel\.parquet

- pipeline\_data/1\.3\.1\.final\_tables/nyc\_taxi\_zones\_harmonized\.parquet

- pipeline\_data/1\.3\.1\.final\_tables/bridge\_tunnel\_mobility\_panel\.parquet \(optional contextual dataset\)

- pipeline\_data/1\.3\.1\.final\_tables/weather\_taxi\_zone\_context\.parquet \(contextual companion dataset for weather\-aware EDA and supervised modeling\)

The Audit\-Only Mobility Panel should only be consulted when validating results or investigating unexpected behavior discovered during EDA or feature engineering\. From this point forward, all analytical work should be based on the Analysis\-Ready Mobility Panel\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>